# 12 — Ajuste curto: Home versus Home com replay de Open Images

O candidato 10 avançou na validação externa e perdeu desempenho em Home.
Este experimento verifica se um ajuste suave recupera Home sem perder esse avanço.

| Braço | Início | Composição de cada batch | Orçamento |
|---|---|---|---|
| A — Home | Candidato 10 | 16 Home | 342 atualizações |
| B — Home + replay | Candidato 10 | 12 Home + 4 Open Images TRAIN | 342 atualizações |

São **3 épocas curtas de 114 batches**, batch 16, AdamW, LR constante de
**0,000003**, FP32, sem warmup e uma atualização por batch. A igualdade é de
atualizações; A recebe mais exposições de Home, e essa diferença será registrada.
Cada época curta do braço misto não percorre todo o replay. Uma única semente
torna este um controle exploratório, sem prova de superioridade geral.

A cada época, o checkpoint EMA será avaliado nas mesmas **456 Home VAL + 1.000
imagens externas** do notebook 11. **As 300 reservadas ficam dentro do ZIP,
sem extração de imagens e sem inferência.** Home TEST e Open Images TEST
também ficam fora do treino e da seleção. Nenhum peso em uso é substituído.


## 1. Preparar o Kaggle

1. Importe este notebook completo.
2. Ative Internet e, de preferência, **GPU T4**. O código usa uma GPU;
   também aceita CPU, com execução mais demorada.
3. Habilite o Secret já existente **GDRIVE_RCLONE_CONFIG_B64**.
4. Use **Save Version → Save & Run All**, começando pela primeira célula.

Os caminhos das execuções 10 e 11 estão preenchidos. Não é necessário enviar
os 100 mil arquivos ao Kaggle: serão usados os **cinco primeiros lotes de treino**
já preparados, com hashes conferidos, para montar replay de 2.000 imagens.
As imagens de treino e validação ficam em /tmp; resultados em /kaggle/working.
Checkpoints e relatórios são copiados ao Drive ao longo da execução.


In [ ]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow scipy matplotlib


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys, json, csv, hashlib, inspect, shutil, subprocess, platform, traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO, settings
from IPython.display import display, FileLink, HTML

VERSAO='12-home-replay-controlado-1.0'
SEED=42
EPOCAS, BATCHES_POR_EPOCA, BATCH, LR0 = 3, 114, 16, 3e-6
DEVICE=0 if torch.cuda.is_available() else 'cpu'
WORKERS=2 if DEVICE==0 else 0
PASTA_EXECUCAO_10='resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010'
PASTA_EXECUCAO_11='resultados/diagnostico11_20261016_024603_683411'
NOME_EXPERIMENTO='ajuste12_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
PASTA_LOCAL=Path('/tmp/tcc_ajuste_12')/NOME_EXPERIMENTO
PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
PASTA_LOCAL.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
if ultralytics.__version__!='8.4.150':
    raise RuntimeError('Reinicie a sessão e execute ultralytics==8.4.150.')
if (EPOCAS,BATCHES_POR_EPOCA,BATCH,LR0)!=(3,114,16,3e-6):
    raise RuntimeError('Este notebook foi preparado para o orçamento fixado; documente mudanças em um novo experimento.')
settings.update({k:False for k in ['sync','hub','platform','wandb','comet','mlflow','neptune','clearml','dvc','tensorboard'] if k in settings})
print('Dispositivo:',torch.cuda.get_device_name(0) if DEVICE==0 else 'CPU')
print('Duas inicializações do candidato 10; 342 atualizações por braço; sem uso dos testes.')


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}}
LOCALIZACAO_11 = {'pasta_drive': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'arquivo': 'validacao_independente_11_695b445e15c9.zip', 'arquivo_sha256': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'dataset_sha256': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'validacao': 1000, 'reserva_futura': 300}


## 2. Código e verificações internas

O avaliador do notebook 11 e a perda parcial usada no 10 estão embutidos.
A versão da implementação da perda é conferida antes do treino. Classes
desconhecidas no Open Images não viram negativos; positivos continuam supervisionados.
Mosaic, MixUp, CutMix e Copy-Paste ficam desativados para preservar essa cobertura.


In [ ]:
MODULOS_EMBUTIDOS = {'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'diag11_reports.py': '"""Comparação pareada, indícios dos erros e galeria de inspeção."""\nfrom __future__ import annotations\n\nimport gzip\nimport html\nimport json\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom diag11_core import (NAMES, LEVEL, alert_on, canonical, ratio, save_csv,\n                         save_json, with_policy)\nfrom diag11_data import stable_order\n\n\ndef paired_bootstrap(old_images, new_images, repetitions=1000, seed=42):\n    old = {r["id"]: r for r in old_images}\n    new = {r["id"]: r for r in new_images}\n    if set(old) != set(new) or len(old) != len(old_images) or len(new) != len(new_images):\n        raise ValueError("Bootstrap exige as mesmas imagens, sem repetições.")\n    ids = sorted(old)\n    fields = ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta", "alta_ref"]\n    a = np.array([[old[i][k] for k in fields] for i in ids], dtype=float)\n    b = np.array([[new[i][k] for k in fields] for i in ids], dtype=float)\n\n    def scores(x):\n        tp, fn, fp, missing, high = x\n        return [100*ratio(tp, tp+fp) if tp+fp else np.nan,\n                100*ratio(tp, tp+fn) if tp+fn else np.nan,\n                100*ratio(2*tp, 2*tp+fp+fn) if 2*tp+fp+fn else np.nan,\n                fn, fp, missing, 100*ratio(missing, high) if high else np.nan]\n\n    point = np.array(scores(b.sum(axis=0))) - np.array(scores(a.sum(axis=0)))\n    rng = np.random.default_rng(seed)\n    results = []\n    for _ in range(repetitions):\n        index = rng.integers(0, len(ids), len(ids))\n        results.append(np.array(scores(b[index].sum(axis=0))) -\n                       np.array(scores(a[index].sum(axis=0))))\n    samples = np.asarray(results)\n    labels = ["precision_alertas_pp", "recall_alertas_pp", "f1_alertas_pp",\n              "fn_alertas", "fp_alertas", "alta_sem_alerta", "taxa_alta_sem_alerta_pp"]\n    rows = []\n    for j, name in enumerate(labels):\n        values = samples[np.isfinite(samples[:, j]), j]\n        low, high = np.quantile(values, [.025, .975]) if len(values) else (None, None)\n        rows.append({"metrica": name, "delta_candidato_menos_antigo": float(point[j]) if np.isfinite(point[j]) else None,\n                     "ic95_inferior": low, "ic95_superior": high,\n                     "reamostragens_validas": len(values), "imagens": len(ids),\n                     "metodo": "Bootstrap pareado por imagem; não corrige possível exposição prévia"})\n    return rows\n\n\ndef comparison_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for name, result in models.items():\n            s, a = result["summary"], result["summary"]["alertas_por_objeto"]\n            rows.append({"coorte": cohort, "modelo": name, "imagens": s["imagens"],\n                         "precision_alertas": a["precision"], "recall_alertas": a["recall"],\n                         "f1_alertas": a["f1"], "tp_alerta": a["tp"], "fn_alerta": a["fn"],\n                         "fp_alerta": a["fp"], "alta_ref": s["alta_ref"],\n                         "alta_sem_alerta": s["alta_sem_alerta"],\n                         "map50_recorte": s["map50_recorte"],\n                         "map50_95_recorte": s["map50_95_recorte"]})\n    return rows\n\n\ndef diagnosis_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for model, result in models.items():\n            events = result["events"]\n            for cid, name in enumerate(NAMES):\n                group = [e for e in events if e["classe_id"] == cid]\n                for indication in sorted({x for e in group for x in e["indicios"].split("|") if x}):\n                    part = [e for e in group if indication in e["indicios"].split("|")]\n                    rows.append({"coorte": cohort, "modelo": model, "classe": name,\n                                 "indicio": indication, "eventos": len(part),\n                                 "fn_alerta": sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                 "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part),\n                                 "observacao": "Indícios podem se sobrepor; não estabelecem a causa do erro."})\n    return rows\n\n\ndef load_prediction_cache(path):\n    records = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            r = json.loads(line)\n            if r["id"] in records:\n                raise ValueError("Cache tem imagem repetida.")\n            records[r["id"]] = r["predicoes"]\n    return records\n\n\ndef select_gallery_cases(old, new, limit=24, seed=42):\n    a, b = {r["id"]: r for r in old}, {r["id"]: r for r in new}\n    if set(a) != set(b):\n        raise ValueError("Galeria exige os dois modelos nas mesmas imagens.")\n    chosen = {}\n\n    def take(label, ordered, number):\n        used = 0\n        for iid in ordered:\n            if iid not in chosen and len(chosen) < limit:\n                chosen[iid] = label\n                used += 1\n                if used == number:\n                    break\n\n    worse = sorted(a, key=lambda i: (-(b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"]),\n                                     -(b[i]["fn_alerta"]-a[i]["fn_alerta"]), stable_order(i, seed)))\n    better = sorted(a, key=lambda i: (b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"],\n                                      b[i]["fn_alerta"]-a[i]["fn_alerta"], stable_order(i, seed)))\n    false = sorted(a, key=lambda i: (-(a[i]["fp_alerta"]+b[i]["fp_alerta"]), stable_order(i, seed)))\n    take("Candidato com mais omissões", [i for i in worse if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) >\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Candidato com menos omissões", [i for i in better if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) <\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Falsos alertas avaliáveis", [i for i in false if a[i]["fp_alerta"]+b[i]["fp_alerta"]], limit//4)\n    take("Amostra por hash, independente do erro", sorted(a, key=lambda i: stable_order(i, seed)), limit)\n    return [{"id": i, "categoria": label} for i, label in chosen.items()]\n\n\ndef font(size):\n    candidates = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n                  "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"]\n    for path in candidates:\n        if Path(path).is_file():\n            return ImageFont.truetype(path, size)\n    return ImageFont.load_default()\n\n\ndef draw_panel(record, objects, title):\n    canvas = Image.new("RGB", (540, 470), "#101a28")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 10), title, fill="#f5f7fa", font=font(18))\n    with Image.open(record["image"]) as image:\n        image = image.convert("RGB")\n        scale = min(520/image.width, 390/image.height)\n        image = image.resize((max(1, round(image.width*scale)),\n                              max(1, round(image.height*scale))), Image.Resampling.LANCZOS)\n    x0, y0 = (540-image.width)//2, 46+(390-image.height)//2\n    canvas.paste(image, (x0, y0))\n    sx, sy = image.width/record["width"], image.height/record["height"]\n    colours = {"alto": "#ff7b8a", "medio": "#ffde70",\n               "baixo": "#6ee7b7", "sem_prioridade": "#adbac7"}\n    for obj in objects:\n        cid = obj["classe_id"]\n        known = bool(record["known_mask"] & (1 << cid))\n        colour = colours[obj["prioridade"]] if known else "#a9afba"\n        x1, y1, x2, y2 = obj["bbox"]\n        box = [x0+x1*sx, y0+y1*sy, x0+x2*sx, y0+y2*sy]\n        draw.rectangle(box, outline=colour, width=2)\n        label = obj["classe"]\n        if "confianca" in obj:\n            label += f" {obj[\'confianca\']:.2f}"\n        label += " / " + obj["prioridade"] if known else " / sem cobertura"\n        # Limita o rótulo à largura visível; não altera a caixa nem as métricas.\n        tx, ty = max(0, min(box[0], 535)), max(44, box[1]-15)\n        text_box = draw.textbbox((tx, ty), label, font=font(11))\n        draw.rectangle(text_box, fill="#101a28")\n        draw.text((tx, ty), label, fill=colour, font=font(11))\n    draw.text((12, 445), "Rosa: alta | amarelo: média | verde: baixa | cinza: sem referência",\n              fill="#c6ced9", font=font(10))\n    return canvas\n\n\ndef render_case(record, old_pred, new_pred, policy, destination):\n    gt = with_policy([g for g in record["gt"] if record["known_mask"] & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    old = with_policy([p for p in old_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    new = with_policy([p for p in new_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    panels = [draw_panel(record, gt, "Referência V2 avaliável"),\n              draw_panel(record, old, "Modelo antigo / confiança 0,25"),\n              draw_panel(record, new, "Candidato / confiança 0,25")]\n    result = Image.new("RGB", (1620, 470), "#101a28")\n    for i, panel in enumerate(panels):\n        result.paste(panel, (540*i, 0))\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    result.save(destination, quality=88)\n    return destination\n\n\ndef format_percent(value):\n    return "sem suporte" if value is None else f"{100*value:.2f}%"\n\n\ndef build_report(output_dir, evaluations, cohorts, audit, bootstrap, policy, dataset_manifest, seed=42):\n    output_dir = Path(output_dir)\n    comparison = comparison_rows(evaluations)\n    diagnosis = diagnosis_rows(evaluations)\n    save_csv(output_dir / "comparacao_modelos.csv", comparison)\n    save_csv(output_dir / "diagnostico_erros.csv", diagnosis)\n    save_csv(output_dir / "bootstrap_pareado.csv", bootstrap)\n    gallery = []\n    for cohort, models in evaluations.items():\n        records = {r["id"]: r for r in cohorts[cohort]}\n        caches = {name: load_prediction_cache(result["cache_file"]) for name, result in models.items()}\n        cases = select_gallery_cases(models["antigo"]["images"], models["candidato"]["images"], seed=seed)\n        old_images = {r["id"]: r for r in models["antigo"]["images"]}\n        new_images = {r["id"]: r for r in models["candidato"]["images"]}\n        for i, case in enumerate(cases):\n            record = records[case["id"]]\n            relative = f"galeria/{cohort}_{i:03}.jpg"\n            render_case(record, caches["antigo"][record["id"]],\n                        caches["candidato"][record["id"]], policy, output_dir/relative)\n            gallery.append({**case, "coorte": cohort, "imagem": record["name"], "arquivo": relative,\n                            "antigo": {k: old_images[record["id"]][k] for k in\n                                       ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "candidato": {k: new_images[record["id"]][k] for k in\n                                          ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "creditos": record.get("creditos", {})})\n    save_json(output_dir / "indice_galeria.json", gallery)\n    new_records = cohorts["openimages_val_nova"]\n    save_csv(output_dir / "revisao_cenarios.csv", [\n        {"id": r["id"], "imagem": r["name"], "ambiente_interno": "nao_revisado",\n         "proximo_do_uso_pretendido": "nao_revisado", "anotacao_parece_correta": "nao_revisado",\n         "observacao": "", "origem": r.get("creditos", {}).get("OriginalLandingURL", "")}\n        for r in new_records])\n    provenance = audit["baseline_original"]["estado_proveniencia_original"]\n    table = "<table><thead><tr><th>Conjunto</th><th>Modelo</th><th>Precisão alertas</th><th>Recall alertas</th><th>F1</th><th>FN</th><th>Alta sem alerta</th></tr></thead><tbody>"\n    for r in comparison:\n        table += "<tr>" + "".join("<td>"+html.escape(str(x))+"</td>" for x in\n                     [r["coorte"], r["modelo"], format_percent(r["precision_alertas"]),\n                      format_percent(r["recall_alertas"]), format_percent(r["f1_alertas"]),\n                      r["fn_alerta"], r["alta_sem_alerta"]]) + "</tr>"\n    table += "</tbody></table>"\n    cards = []\n    for case in gallery:\n        credit = case["creditos"]\n        source_url = str(credit.get("OriginalLandingURL", ""))\n        license_url = str(credit.get("License", ""))\n        attribution = html.escape(str(credit.get("Author", "")))\n        if source_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(source_url, quote=True)+\'">Fonte original</a>\'\n        if license_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(license_url, quote=True)+\'">Licença</a>\'\n        details = (f"Antigo: FN {case[\'antigo\'][\'fn_alerta\']}, FP {case[\'antigo\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'antigo\'][\'alta_sem_alerta\']}. "\n                   f"Candidato: FN {case[\'candidato\'][\'fn_alerta\']}, FP {case[\'candidato\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'candidato\'][\'alta_sem_alerta\']}.")\n        cards.append("<article><h3>"+html.escape(case["coorte"]+" / "+case["categoria"])+\n                     "</h3><p>"+html.escape(case["imagem"]+" — "+details)+"</p>"+\n                     \'<img loading="lazy" src="\'+case["arquivo"]+\'" alt="Caixas de referência e dos dois modelos">\'+\n                     "<p class=small>"+attribution+\n                     " · Sobreposição de caixas para avaliação.</p></article>")\n    html_report = """<!doctype html><html lang="pt-BR"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Diagnóstico dos detectores</title><style>\nbody{font:16px/1.6 system-ui,sans-serif;background:#f3f5f7;color:#142135;margin:0}\nmain{max-width:1400px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}\n.note,article,table{background:white;border:1px solid #dbe1e8;border-radius:12px}\n.note,article{padding:20px;margin:20px 0}table{border-collapse:collapse;width:100%}\ntd,th{padding:10px;text-align:left;border-bottom:1px solid #e6eaf0}th{background:#e8eff8}\nimg{max-width:100%;height:auto}.small{font-size:12px;color:#526174}.scroll{overflow:auto}\n</style><main><h1>Diagnóstico comparativo dos detectores</h1>\n<p>Pesos congelados · 12 classes · política V2 · confiança 0,25 · sem treinamento</p>\n<div class=note><strong>Como interpretar</strong><p>\nA Home VAL é histórica e exploratória. A validação externa é nova em relação aos IDs\ne bytes dos dados conhecidos do projeto; pré-treinamentos públicos não foram auditados.\nO recorte Open Images tem cobertura parcial e não garante cenas internas.\nAs métricas de alertas medem concordância com a V2, não risco físico independente.\nA reserva futura não foi inferida. Nenhum modelo foi promovido automaticamente.</p>\n<p>Proveniência documental do baseline: <strong>""" + html.escape(provenance) + """</strong>.</p></div>\n<h2>Alertas por objeto avaliável</h2><div class=scroll>""" + table + """</div>\n<p>Os CSVs trazem suporte por classe, AP no recorte, contagens e intervalos pareados.\nNão compare APs de protocolos e distribuições diferentes como se fossem o mesmo benchmark.</p>\n<h2>Casos para inspeção</h2><p>Os casos abaixo priorizam diferenças e erros, além de uma amostra por hash.\nEsta galeria não é uma estimativa da frequência dos erros. As contagens acima usam todas as imagens.\nObjetos sem cobertura aparecem em cinza e não são classificados automaticamente como falsos positivos.</p>\n""" + "".join(cards) + """<h2>Próxima decisão</h2><p>\nRevisar os cenários e os indícios dos erros; só depois definir o treino de controle.\nOs pesos antigos, o candidato e os testes anteriores permanecem preservados.</p></main></html>"""\n    (output_dir / "relatorio_diagnostico.html").write_text(html_report, encoding="utf-8")\n    lines = ["# Diagnóstico 11", "", "Comparação sem novo treinamento e sem seleção automática.",\n             "", f"Proveniência original do baseline: {provenance}.",\n             f"Validação externa: {len(new_records)} imagens.",\n             f"Reserva futura não inferida: {dataset_manifest[\'contagens\'].get(\'reserva_futura\', 0)} imagens.",\n             "", "| Conjunto | Modelo | Precisão alertas | Recall alertas | F1 | FN | Alta sem alerta |",\n             "|---|---|---:|---:|---:|---:|---:|"]\n    for r in comparison:\n        lines.append("| "+" | ".join(map(str, [r["coorte"], r["modelo"],\n                     format_percent(r["precision_alertas"]), format_percent(r["recall_alertas"]),\n                     format_percent(r["f1_alertas"]), r["fn_alerta"], r["alta_sem_alerta"]]))+" |")\n    lines += ["", "A nova cobertura é conservadora e a seleção busca diversidade de classes; não representa a frequência natural do uso.",\n              "A Home VAL continua exploratória enquanto a origem do primeiro treinamento não estiver esclarecida.",\n              "Semelhança por pHash é um filtro de suspeitas, não uma prova de identidade de cena.",\n              "Os indícios de erro podem se sobrepor e não identificam causas de forma conclusiva.",\n              "A reserva atual não foi inferida; não há avaliação independente de risco físico.",\n              "", "Veja relatorio_diagnostico.html para a galeria e bootstrap_pareado.csv para a incerteza das diferenças."]\n    (output_dir / "relatorio_diagnostico.md").write_text("\\n".join(lines)+"\\n", encoding="utf-8")\n    return comparison, diagnosis, gallery\n\n\ndef package_reports(output_dir):\n    output_dir = Path(output_dir)\n    archive = output_dir / "pacote_diagnostico_11.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(output_dir.rglob("*")):\n            if path.is_file() and path != archive and not path.name.endswith(".tmp"):\n                z.write(path, path.relative_to(output_dir))\n    return archive\n', 'exp10_partial.py': '"""Experimento 09: caixas humanas parciais, fontes equilibradas e Home VAL."""\nfrom __future__ import annotations\n\nimport hashlib\nimport inspect\nimport io\nimport json\nimport math\nimport shutil\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport ultralytics\nimport yaml\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Sampler\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\nfrom ultralytics.utils.loss import v8DetectionLoss\nfrom ultralytics.utils.torch_utils import unwrap_model\n\n\nVERSION = "10-openimages-80-20-1ep-3.0"\nULTRALYTICS_VERSION = "8.4.150"\nLOSS_SOURCE_SHA = "357053aa9a24ad2c4d541c3ada3a27232a10d679f89c7b4957b9a84a7da4ec1d"\n\nNOMES = [\n    "bed", "sofa", "chair", "table", "lamp", "tv",\n    "laptop", "wardrobe", "window", "door",\n    "potted plant", "photo frame"\n]\n\nALL_KNOWN = (1 << 12) - 1\n\n\ndef sha256_file(p):\n    h = hashlib.sha256()\n\n    with Path(p).open("rb") as f:\n        for b in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            h.update(b)\n\n    return h.hexdigest()\n\n\ndef canonical_json(x):\n    return json.dumps(\n        x,\n        sort_keys=True,\n        ensure_ascii=False,\n        separators=(",", ":")\n    )\n\n\ndef save_json(p, x):\n    p = Path(p)\n    p.parent.mkdir(parents=True, exist_ok=True)\n\n    tmp = p.with_name(p.name + ".tmp")\n    tmp.write_text(\n        json.dumps(x, ensure_ascii=False, indent=2),\n        encoding="utf-8"\n    )\n    tmp.replace(p)\n\n\ndef check_api():\n    if ultralytics.__version__ != ULTRALYTICS_VERSION:\n        raise RuntimeError(\n            f"Instale ultralytics=={ULTRALYTICS_VERSION} "\n            "e reinicie o ambiente se necessário."\n        )\n\n    if sha256_file(inspect.getfile(v8DetectionLoss)) != LOSS_SOURCE_SHA:\n        raise RuntimeError(\n            "A implementação da perda difere da versão verificada; "\n            "interrompido antes do treino."\n        )\n\n\nclass BCEComCobertura(nn.Module):\n    """Zera perdas negativas de classes desconhecidas; conserva positivos."""\n\n    def __init__(self, original=None):\n        super().__init__()\n        self.original = original or nn.BCEWithLogitsLoss(reduction="none")\n        self.known = None\n\n    def forward(self, logits, target):\n        if self.known is None:\n            raise RuntimeError(\n                "Cobertura não foi transmitida à perda parcial."\n            )\n\n        known = self.known.to(\n            device=logits.device,\n            dtype=torch.bool\n        )\n\n        if known.shape != (logits.shape[0], logits.shape[-1]):\n            raise RuntimeError(\n                "Dimensões de cobertura incompatíveis com batch/classes."\n            )\n\n        permitted = known[:, None, :] | (target > 0)\n\n        return self.original(logits, target) * permitted.to(logits.dtype)\n\n\nclass DetectionLossParcial(v8DetectionLoss):\n    """Altera somente a BCE; caixa, DFL, assignment e ganhos são os padrões."""\n\n    def __init__(self, model):\n        super().__init__(model)\n\n        if (\n            self.nc != 12\n            or not isinstance(self.bce, nn.BCEWithLogitsLoss)\n            or self.bce.reduction != "none"\n        ):\n            raise RuntimeError(\n                "Cabeça ou perda de detecção não é a esperada "\n                "para este experimento."\n            )\n\n        self.bce = BCEComCobertura(self.bce)\n\n    def loss(self, preds, batch):\n        if "class_known" not in batch:\n            raise RuntimeError(\n                "Batch de treinamento sem cobertura; "\n                "não substituir por negativos presumidos."\n            )\n\n        self.bce.known = batch["class_known"]\n\n        try:\n            return super().loss(preds, batch)\n        finally:\n            self.bce.known = None\n\n\nclass SamplerDuasFontes(Sampler):\n    """Cada batch contém metade Home e metade OI; uma época cobre todos os OI."""\n\n    def __init__(self, home_indices, oi_indices, batch_size, seed=42):\n        self.home = list(home_indices)\n        self.oi = list(oi_indices)\n\n        if (\n            not self.home\n            or not self.oi\n            or batch_size < 2\n            or batch_size % 2\n        ):\n            raise ValueError(\n                "São necessárias duas fontes e um batch par >=2."\n            )\n\n        self.batch = int(batch_size)\n        self.half = self.batch // 2\n        self.batches = math.ceil(len(self.oi) / self.half)\n        self.seed = int(seed)\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def __len__(self):\n        return self.batches * self.batch\n\n    def __iter__(self):\n        rng = np.random.default_rng(self.seed + self.epoch)\n\n        home, oi = [], []\n        needed = self.batches * self.half\n\n        while len(home) < needed:\n            home.extend(rng.permutation(self.home).tolist())\n\n        while len(oi) < needed:\n            oi.extend(rng.permutation(self.oi).tolist())\n\n        for start in range(0, needed, self.half):\n            batch = (\n                home[start:start + self.half]\n                + oi[start:start + self.half]\n            )\n            rng.shuffle(batch)\n            yield from batch\n\n\nclass LoaderFinito(DataLoader):\n    def reset(self):\n        # Um novo iterador é criado em cada época.\n        pass\n\n\ndef epoch_sampler(trainer):\n    sampler = getattr(trainer.train_loader, "sampler", None)\n\n    if isinstance(sampler, SamplerDuasFontes):\n        sampler.set_epoch(trainer.epoch)\n\n\nclass TrainerOpenImages(DetectionTrainer):\n    # Configurado antes de criar o trainer.\n    # Não é anexado ao DetectionModel salvo.\n    source_records = None\n\n    def __init__(self, *args, **kwargs):\n        check_api()\n        super().__init__(*args, **kwargs)\n        self.add_callback("on_train_epoch_start", epoch_sampler)\n\n    def get_dataloader(\n        self,\n        dataset_path,\n        batch_size=16,\n        rank=0,\n        mode="train"\n    ):\n        if mode != "train":\n            return super().get_dataloader(\n                dataset_path,\n                batch_size,\n                rank,\n                mode\n            )\n\n        if self.source_records is None:\n            raise RuntimeError(\n                "As fontes/coberturas ainda não foram configuradas."\n            )\n\n        if self.world_size > 1 or self.args.compile or self.args.rect:\n            raise RuntimeError(\n                "Este experimento usa uma GPU, "\n                "compile=False e rect=False."\n            )\n\n        for k in ["mosaic", "mixup", "cutmix", "copy_paste"]:\n            if float(getattr(self.args, k, 0)) != 0:\n                raise RuntimeError(\n                    f"{k} deve ficar em zero para preservar "\n                    "a cobertura por imagem."\n                )\n\n        dataset = self.build_dataset(\n            dataset_path,\n            mode,\n            batch_size\n        )\n\n        home, oi = [], []\n\n        for i, arquivo in enumerate(dataset.im_files):\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    f"Imagem sem proveniência/cobertura: {arquivo}"\n                )\n\n            source = self.source_records[key]["source"]\n\n            if source == "home":\n                home.append(i)\n            elif source == "openimages":\n                oi.append(i)\n            else:\n                raise RuntimeError(\n                    "Fonte de treinamento não reconhecida."\n                )\n\n        sampler = SamplerDuasFontes(\n            home,\n            oi,\n            batch_size,\n            self.args.seed\n        )\n\n        generator = torch.Generator().manual_seed(self.args.seed)\n\n        return LoaderFinito(\n            dataset,\n            batch_size=batch_size,\n            sampler=sampler,\n            num_workers=self.args.workers,\n            pin_memory=self.device.type == "cuda",\n            collate_fn=dataset.collate_fn,\n            generator=generator,\n            drop_last=False\n        )\n\n    def preprocess_batch(self, batch):\n        batch = super().preprocess_batch(batch)\n\n        masks = []\n\n        for arquivo in batch["im_file"]:\n            key = Path(arquivo).resolve().as_posix()\n\n            if key not in self.source_records:\n                raise RuntimeError(\n                    "Batch contém imagem fora do manifesto de treino."\n                )\n\n            bitmask = self.source_records[key]["known_mask"]\n\n            masks.append([\n                bool(bitmask & (1 << cid))\n                for cid in range(12)\n            ])\n\n        batch["class_known"] = torch.tensor(\n            masks,\n            dtype=torch.bool,\n            device=self.device\n        )\n\n        model = unwrap_model(self.model)\n        criterion = getattr(model, "criterion", None)\n\n        if criterion is None:\n            # Modelo padrão, já colocado no dispositivo.\n            # A EMA foi criada antes desta etapa e valida Home\n            # com a perda padrão de cobertura completa.\n            model.criterion = DetectionLossParcial(model)\n\n        elif not isinstance(criterion, DetectionLossParcial):\n            raise RuntimeError(\n                "O treino iniciou com uma perda diferente "\n                "da perda parcial."\n            )\n\n        return batch\n\n\ndef self_test_bce():\n    logits = torch.zeros(\n        (2, 3, 12),\n        requires_grad=True\n    )\n\n    targets = torch.zeros_like(logits)\n    targets[1, 0, 7] = 1\n\n    known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n    known[1, 7] = False\n    known[1, 10] = False\n\n    partial = BCEComCobertura()\n    partial.known = known\n    partial(logits, targets).sum().backward()\n\n    assert logits.grad[1, :, 10].eq(0).all()\n    assert logits.grad[1, 1:, 7].eq(0).all()\n    assert logits.grad[1, 0, 7].item() < 0\n    assert logits.grad[0].ne(0).all()\n\n    full = BCEComCobertura()\n    full.known = torch.ones(\n        (2, 12),\n        dtype=torch.bool\n    )\n\n    torch.testing.assert_close(\n        full(logits.detach(), targets),\n        nn.BCEWithLogitsLoss(reduction="none")(\n            logits.detach(),\n            targets\n        )\n    )\n\n    return {\n        "gradiente_negativo_desconhecido_zero": True,\n        "caixa_positiva_preservada": True,\n        "cobertura_completa_igual_bce_padrao": True\n    }\n\n\ndef read_safe(z, name):\n    path = PurePosixPath(name)\n\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name:\n        raise RuntimeError(\n            f"Caminho inseguro dentro de pacote: {name}"\n        )\n\n    return z.read(name)\n\n\ndef validate_yolo(raw, nclasses=12, source="<anotação>"):\n    """\n    Ignora linhas vazias e valida as linhas preenchidas.\n    Em caso de erro, informa arquivo, linha e conteúdo.\n    """\n    result = []\n\n    try:\n        text = raw.decode("utf-8-sig")\n    except UnicodeDecodeError as exc:\n        raise RuntimeError(\n            f"Anotação YOLO não está em UTF-8: {source}"\n        ) from exc\n\n    for line_number, line in enumerate(text.splitlines(), 1):\n        if not line.strip():\n            continue\n\n        values = line.split()\n        location = f"{source}, linha {line_number}"\n\n        if len(values) != 5:\n            raise RuntimeError(\n                f"Anotação YOLO inválida em {location}: "\n                "esperados 5 campos "\n                "(classe x y largura altura), "\n                f"encontrados {len(values)}. "\n                f"Conteúdo: {line[:160]!r}"\n            )\n\n        try:\n            cid, *bbox = map(float, values)\n        except ValueError as exc:\n            raise RuntimeError(\n                f"Campo não numérico em {location}: "\n                f"{line[:160]!r}"\n            ) from exc\n\n        if (\n            not cid.is_integer()\n            or not 0 <= cid < nclasses\n            or not all(math.isfinite(x) for x in bbox)\n        ):\n            raise RuntimeError(\n                f"Classe ou caixa YOLO inválida em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        x, y, w, h = bbox\n\n        if not (\n            0 <= x <= 1\n            and 0 <= y <= 1\n            and 0 < w <= 1\n            and 0 < h <= 1\n            and x - w / 2 >= -1e-6\n            and x + w / 2 <= 1 + 1e-6\n            and y - h / 2 >= -1e-6\n            and y + h / 2 <= 1 + 1e-6\n        ):\n            raise RuntimeError(\n                f"Caixa YOLO fora dos limites em {location}: "\n                f"{line[:160]!r}"\n            )\n\n        result.append(int(cid))\n\n    if not result:\n        raise RuntimeError(\n            f"Imagem desta preparação sem caixas: {source}"\n        )\n\n    return result\n\n\ndef write_checked(dest, raw, expected_sha):\n    if hashlib.sha256(raw).hexdigest() != expected_sha:\n        raise RuntimeError(\n            "Conteúdo do pacote diferente do SHA-256 no manifesto."\n        )\n\n    dest = Path(dest)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    dest.write_bytes(raw)\n\n\ndef prepare_training(\n    zip_home,\n    oi_drive,\n    local,\n    results,\n    expected,\n    batches_oi=None,\n    home_train_count=1824,\n    home_val_count=456,\n    oi_count=100000,\n    package_fetcher=None\n):\n    """Snapshot Home TRAIN/VAL e de todos os lotes liberados para 100k."""\n\n    local = Path(local)\n    results = Path(results)\n    oi_drive = Path(oi_drive)\n\n    local.mkdir(parents=True, exist_ok=True)\n    results.mkdir(parents=True, exist_ok=True)\n\n    prep_cfg = json.loads(\n        (oi_drive / "configuracao_preparacao.json").read_text()\n    )\n\n    cfg_sha = hashlib.sha256(\n        canonical_json(prep_cfg).encode()\n    ).hexdigest()\n\n    if (\n        cfg_sha != expected["oi_config_sha256"]\n        or prep_cfg["nomes"] != NOMES\n    ):\n        raise RuntimeError(\n            "Preparação Open Images diferente da auditada no piloto."\n        )\n\n    home_zip_local = local / "home_sun_snapshot.zip"\n    shutil.copyfile(zip_home, home_zip_local)\n\n    if sha256_file(home_zip_local) != expected["home_zip_sha256"]:\n        raise RuntimeError(\n            "ZIP Home+SUN diferente do preservado no 07b/08."\n        )\n\n    records, home_val = {}, []\n    extract_home = local / "base_existente"\n\n    with zipfile.ZipFile(home_zip_local) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError(\n                "ZIP Home+SUN tem nomes repetidos."\n            )\n\n        manifest = json.loads(\n            read_safe(z, "verificacao_preparacao.json")\n        )\n\n        if (\n            manifest["test"]["imagens"] != 403\n            or manifest["test"]["sha256"] != expected["test_sha256"]\n        ):\n            raise RuntimeError(\n                "TEST preservado diferente do esperado."\n            )\n\n        if (\n            {int(k): v for k, v in manifest["classes"].items()}\n            != dict(enumerate(NOMES))\n        ):\n            raise RuntimeError(\n                "Classes/IDs da base Home incompatíveis."\n            )\n\n        root = manifest["dataset_dir"]\n\n        for item in manifest["arquivos"]:\n            name = item["arquivo"]\n            p = PurePosixPath(name)\n\n            in_home_train = (\n                name.startswith(f"{root}/images/train/raw_train/")\n                or name.startswith(f"{root}/labels/train/raw_train/")\n            )\n\n            in_home_val = (\n                name.startswith(f"{root}/images/val/raw_train/")\n                or name.startswith(f"{root}/labels/val/raw_train/")\n            )\n\n            code_or_meta = (\n                name.startswith("codigo/")\n                or (\n                    p.suffix in {\n                        ".json", ".csv", ".yaml", ".yml", ".txt"\n                    }\n                    and "/images/" not in name\n                    and "/labels/" not in name\n                )\n            )\n\n            if not (in_home_train or in_home_val or code_or_meta):\n                continue\n\n            raw = read_safe(z, name)\n            dest = extract_home / name\n            write_checked(dest, raw, item["sha256"])\n\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                records[dest.resolve().as_posix()] = {\n                    "source": "home",\n                    "known_mask": ALL_KNOWN,\n                    "sha256": item["sha256"]\n                }\n\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                home_val.append(\n                    dest.resolve().as_posix()\n                )\n\n        listed = {\n            item["arquivo"]\n            for item in manifest["arquivos"]\n        }\n\n        # O hash do ZIP também autentica o código auxiliar\n        # que não esteja listado individualmente no manifesto.\n        for name in set(z.namelist()) - listed:\n            if name.startswith("codigo/") and not name.endswith("/"):\n                raw = read_safe(z, name)\n\n                write_checked(\n                    extract_home / name,\n                    raw,\n                    hashlib.sha256(raw).hexdigest()\n                )\n\n    home_dataset = extract_home / root\n    home_images = list(records)\n\n    if (\n        len(home_images) != home_train_count\n        or len(home_val) != home_val_count\n    ):\n        raise RuntimeError(\n            "Home TRAIN/VAL inesperados: "\n            f"{len(home_images)}/{len(home_val)}; "\n            f"esperados {home_train_count}/{home_val_count}."\n        )\n\n    for image in [*home_images, *home_val]:\n        label = Path(\n            image.replace("/images/", "/labels/")\n        ).with_suffix(".txt")\n\n        validate_yolo(\n            label.read_bytes(),\n            source=str(label)\n        )\n\n    packages = json.loads(\n        (oi_drive / "manifesto_pacotes.json").read_text()\n    )\n\n    if packages["config_sha256"] != cfg_sha:\n        raise RuntimeError(\n            "Manifesto dos lotes tem configuração diferente."\n        )\n\n    selected = sorted(\n        packages["lotes"],\n        key=lambda r: r["arquivo"]\n    )\n    if batches_oi is None:\n        batches_oi = len(selected)\n\n    if len(selected) != batches_oi:\n        raise RuntimeError(\n            "Quantidade de lotes difere da liberação para 100k."\n        )\n\n    oi_ids, oi_hashes, package_snapshot = set(), set(), []\n    oi_root = local / "openimages"\n    per_class = Counter()\n\n    for item in selected:\n        name = item["arquivo"]\n\n        if PurePosixPath(name).name != name:\n            raise RuntimeError(\n                "Nome de pacote não é um nome simples."\n            )\n\n        snapshot = local / name\n\n        if package_fetcher is None:\n            shutil.copyfile(oi_drive / "lotes" / name, snapshot)\n        else:\n            print(f"Pacote {len(package_snapshot) + 1}/{len(selected)}: {name}", flush=True)\n            package_fetcher(item, snapshot)\n\n        if (\n            sha256_file(snapshot) != item["zip_sha256"]\n            or snapshot.stat().st_size != item["zip_bytes"]\n        ):\n            raise RuntimeError(\n                f"Pacote alterado ou incompleto: {name}"\n            )\n\n        with zipfile.ZipFile(snapshot) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError(\n                    "ZIP Open Images tem membros repetidos."\n                )\n\n            m = json.loads(\n                read_safe(z, "manifesto_lote.json")\n            )\n\n            if (\n                m["config_sha256"] != cfg_sha\n                or m["origem_split"] != "train"\n            ):\n                raise RuntimeError(\n                    "Lote não é da seleção do treino "\n                    "Open Images auditado."\n                )\n\n            for r in m["aceitas"]:\n                if r["id"] in oi_ids or r["sha256"] in oi_hashes:\n                    raise RuntimeError(\n                        "ID ou bytes duplicados nos lotes usados no treino."\n                    )\n\n                oi_ids.add(r["id"])\n                oi_hashes.add(r["sha256"])\n\n                im_raw = read_safe(z, r["imagem"])\n                label_raw = read_safe(z, r["label"])\n                cov_raw = read_safe(z, r["cobertura"])\n\n                for key, raw, hkey in [\n                    ("imagem", im_raw, "sha256"),\n                    ("label", label_raw, "label_sha256"),\n                    ("cobertura", cov_raw, "cobertura_sha256")\n                ]:\n                    write_checked(\n                        oi_root / r[key],\n                        raw,\n                        r[hkey]\n                    )\n\n                cls = validate_yolo(\n                    label_raw,\n                    source=f"{name}:{r[\'label\']}"\n                )\n\n                cov = json.loads(cov_raw)\n\n                if (\n                    cov["known_mask"] != r["known_mask"]\n                    or not isinstance(r["known_mask"], int)\n                    or not 0 <= r["known_mask"] <= ALL_KNOWN\n                ):\n                    raise RuntimeError(\n                        "Cobertura do lote inconsistente."\n                    )\n\n                known_list = (\n                    set(cov["classes_positivas_verificadas_exaustivas"])\n                    | set(cov["classes_negativas_verificadas"])\n                )\n\n                if sum(1 << cid for cid in known_list) != r["known_mask"]:\n                    raise RuntimeError(\n                        "Lista de classes conhecidas "\n                        "não corresponde à máscara."\n                    )\n\n                if set(cls) & set(cov["classes_negativas_verificadas"]):\n                    raise RuntimeError(\n                        "Caixa positiva contradiz ausência verificada."\n                    )\n\n                with Image.open(io.BytesIO(im_raw)) as im:\n                    im.load()\n\n                    if im.size != (r["width"], r["height"]):\n                        raise RuntimeError(\n                            "Dimensões da imagem diferem do manifesto."\n                        )\n\n                image = (\n                    oi_root / r["imagem"]\n                ).resolve().as_posix()\n\n                records[image] = {\n                    "source": "openimages", "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "oi_id": r["id"], "lote": name,\n                    "classe_primaria": int(r.get("classe_primaria_selecao", min(cls))),\n                    "classes_com_caixa": cls, "width": r["width"], "height": r["height"],\n                    "label_sha256": r["label_sha256"], "cobertura_sha256": r["cobertura_sha256"],\n                }\n\n                per_class.update(cls)\n\n        package_snapshot.append(item)\n\n        # Remove apenas a cópia local do ZIP do lote.\n        snapshot.unlink()\n\n    if len(oi_ids) != oi_count:\n        raise RuntimeError(\n            f"Treinamento esperava {oi_count} imagens OI, "\n            f"encontrou {len(oi_ids)}."\n        )\n\n    if set(home_val) & set(records):\n        raise RuntimeError(\n            "Imagem de validação aparece no treino."\n        )\n\n    val_hashes = {\n        sha256_file(x)\n        for x in home_val\n    }\n\n    if val_hashes & {\n        r["sha256"]\n        for r in records.values()\n    }:\n        raise RuntimeError(\n            "Imagem de treino coincide em bytes "\n            "com a validação Home."\n        )\n\n    # O 08 já realizou a triagem de contaminação.\n    # Aqui repetimos a verificação por bytes contra Home VAL.\n    train_list = local / "train.txt"\n    val_list = local / "home_val.txt"\n\n    train_list.write_text(\n        "\\n".join(sorted(records)) + "\\n",\n        encoding="utf-8"\n    )\n\n    val_list.write_text(\n        "\\n".join(sorted(home_val)) + "\\n",\n        encoding="utf-8"\n    )\n\n    data_yaml = local / "data_home_openimages.yaml"\n\n    data_yaml.write_text(\n        yaml.safe_dump(\n            {\n                "path": str(local.resolve()),\n                "train": str(train_list),\n                "val": str(val_list),\n                "names": dict(enumerate(NOMES))\n            },\n            sort_keys=False,\n            allow_unicode=True\n        ),\n        encoding="utf-8"\n    )\n\n    save_json(\n        local / "fontes_cobertura_treino.json",\n        records\n    )\n\n    summary = {\n        "versao": VERSION,\n        "home_train_imagens_unicas": len(home_images),\n        "home_val_imagens": len(home_val),\n        "openimages_imagens_unicas": len(oi_ids),\n        "treino_imagens_unicas": len(records),\n        "caixas_openimages": sum(per_class.values()),\n        "caixas_openimages_por_classe": dict(per_class),\n        "lotes_utilizados": package_snapshot,\n        "home_zip_sha256": expected["home_zip_sha256"],\n        "test_sha256": expected["test_sha256"],\n        "test_imagens": 403,\n        "test_extraido_ou_avaliado": False,\n        "sun_incluida_no_novo_treino": False,\n        "amostragem": (\n            "50% Home / 50% OI em cada batch; "\n            "Home reamostrado por índices, sem duplicar arquivos"\n        ),\n        "selecao_checkpoint": (\n            "mAP50-95 Home VAL; triagem de alertas após treino"\n        ),\n        "tratamento_parcial": (\n            "Zero gradiente BCE para negativos de classes desconhecidas; "\n            "caixas positivas preservadas"\n        ),\n        "config_preparacao_sha256": cfg_sha\n    }\n\n    save_json(\n        results / "resumo_dados_treino.json",\n        summary\n    )\n\n    save_json(\n        results / "lotes_utilizados.json",\n        package_snapshot\n    )\n\n    TrainerOpenImages.source_records = records\n\n    return {\n        "yaml": data_yaml,\n        "home_dataset": home_dataset,\n        "codigo": extract_home / "codigo",\n        "records": records,\n        "home_val": home_val,\n        "resumo": summary\n    }\n\n\ndef export_standard_checkpoint(source, dest):\n    """Confere que o checkpoint não depende das classes do treinador parcial."""\n\n    obj = torch.load(\n        source,\n        map_location="cpu",\n        weights_only=False\n    )\n\n    model = obj.get("ema") or obj.get("model")\n\n    if type(model) is not DetectionModel:\n        raise RuntimeError(\n            "O checkpoint não contém DetectionModel padrão."\n        )\n\n    model.criterion = None\n    model = model.float().eval()\n\n    if not all(\n        torch.isfinite(t).all()\n        for t in model.state_dict().values()\n        if t.is_floating_point()\n    ):\n        raise RuntimeError(\n            "Checkpoint com pesos não finitos."\n        )\n\n    obj["model"] = model\n    obj["ema"] = None\n    obj["optimizer"] = None\n\n    torch.save(obj, dest)\n\n    return {\n        "arquivo": str(dest),\n        "sha256": sha256_file(dest),\n        "modelo_padrao": True,\n        "criterion_serializada": False\n    }', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'exp12_data.py': '"""Entradas congeladas e replay somente do TRAIN histórico."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport yaml\n\nimport diag11_core as core\nimport diag11_data as d11\nfrom exp12_protocol import diverse_subset\n\n\ndef check_digest(raw, expected, label):\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise RuntimeError("Conteúdo alterado: " + label)\n\n\ndef materialise_home_labels(zip_path, home):\n    with zipfile.ZipFile(zip_path) as z:\n        for record in home:\n            member = record["id"].removeprefix("home:")\n            label_name = member.replace("/images/", "/labels/").rsplit(".", 1)[0]+".txt"\n            core.safe_member(label_name)\n            raw = z.read(label_name)\n            check_digest(raw, record["label_sha256"], label_name)\n            label = Path(str(record["image"]).replace("/images/", "/labels/")).with_suffix(".txt")\n            label.parent.mkdir(parents=True, exist_ok=True)\n            label.write_bytes(raw)\n            record["label"] = label\n\n\ndef read_package11(path, expected_sha, expected_models, expected_dataset):\n    if core.sha256_file(path) != expected_sha:\n        raise RuntimeError("O pacote do diagnóstico 11 mudou.")\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Pacote 11 com membros repetidos.")\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read("resumo_diagnostico_11.json"))\n        config = json.loads(z.read("configuracao_diagnostico.json"))\n        dataset_location = json.loads(z.read("localizacao_dataset.json"))\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if (config["modelos_sha256"] != expected_models or\n                config["dataset_sha256"] != expected_dataset or\n                dataset_location["dataset_sha256"] != expected_dataset or\n                manifest["dataset_sha256"] != expected_dataset or\n                summary["reserva_futura_inferida"] is not False):\n            raise RuntimeError("Identidades ou reserva do diagnóstico 11 incompatíveis.")\n    return summary, config, dataset_location, manifest\n\n\ndef load_frozen_validation(zip_path, destination, expected_archive_sha, expected_dataset_sha,\n                           expected_manifest):\n    if core.sha256_file(zip_path) != expected_archive_sha:\n        raise RuntimeError("ZIP da validação independente diferente do congelado no 11.")\n    destination = Path(destination)\n    val, reserved_ids, frozen_files = [], set(), []\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Dataset novo com membros repetidos.")\n        for member in z.namelist():\n            core.safe_member(member)\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if manifest != expected_manifest:\n            raise RuntimeError("O manifesto do pacote difere do diagnóstico 11.")\n        raw_manifest = dict(manifest)\n        claimed = raw_manifest.pop("dataset_sha256")\n        if hashlib.sha256(core.canonical(raw_manifest).encode()).hexdigest() != claimed or claimed != expected_dataset_sha:\n            raise RuntimeError("Hash canônico da validação nova não confere.")\n        if manifest["nomes"] != core.NAMES or manifest["origem_split"] != "validation":\n            raise RuntimeError("Classes ou origem da validação alteradas.")\n        items = manifest["arquivos"]\n        if (Counter(r["role"] for r in items) != {"val_nova": 1000, "reserva_futura": 300}\n                or len({r["id"] for r in items}) != 1300\n                or len({r["sha256"] for r in items}) != 1300):\n            raise RuntimeError("Identidades ou quantidades do conjunto independente incompatíveis.")\n        for item in items:\n            role = item["role"]\n            for field, prefix in [("image", "images"), ("label", "labels"), ("coverage", "coverage")]:\n                if not item[field].startswith(prefix+"/"+role+"/"):\n                    raise RuntimeError("Caminho do dataset não corresponde ao papel declarado.")\n                if item[field] not in z.namelist():\n                    raise RuntimeError("Arquivo declarado ausente no ZIP.")\n            if role == "reserva_futura":\n                reserved_ids.add(item["id"])\n                continue  # Nenhum JPEG, label ou cobertura da reserva é extraído.\n            paths = {}\n            for field, hash_field in [("image", "sha256"), ("label", "label_sha256"),\n                                      ("coverage", "coverage_sha256")]:\n                raw = z.read(item[field])\n                check_digest(raw, item[hash_field], item[field])\n                output = destination/item[field]\n                output.parent.mkdir(parents=True, exist_ok=True)\n                output.write_bytes(raw)\n                paths[field] = output\n                frozen_files.append((output, item[hash_field]))\n            coverage = json.loads(paths["coverage"].read_text())\n            if any(coverage[k] != item[k] for k in ["known_mask", "positive_mask", "negative_mask"]):\n                raise RuntimeError("Cobertura da validação foi alterada.")\n            with Image.open(paths["image"]) as image:\n                if image.size != (item["width"], item["height"]):\n                    raise RuntimeError("Dimensões diferentes do manifesto.")\n                image.verify()\n            parsed = core.parse_yolo(paths["label"].read_bytes(), item["width"], item["height"])\n            if len(parsed) != len(item["gt_normalizado"]):\n                raise RuntimeError("Quantidade de caixas difere do manifesto.")\n            gt = []\n            for original, label_obj in zip(item["gt_normalizado"], parsed):\n                a, b, c, d = original["bbox_norm"]\n                box = [a*item["width"], b*item["height"], c*item["width"], d*item["height"]]\n                if (original["classe_id"] != label_obj["classe_id"] or\n                        not np.allclose(box, label_obj["bbox"], rtol=0, atol=.001)):\n                    raise RuntimeError("Caixa do manifesto diverge do label autenticado.")\n                gt.append({**original, "bbox": box})\n            val.append({**item, **paths, "gt": gt, "role": "val_nova",\n                        "coorte": "openimages_val_nova", "name": paths["image"].name})\n    return val, reserved_ids, frozen_files\n\n\ndef load_replay(package_items, selected_history, local_dir, fetch_package, expected_cfg,\n                forbidden_ids, forbidden_hashes, protected_near, validator, final_size=2000, seed=42):\n    """Lê só os membros escolhidos dos cinco lotes existentes, com SHA integral."""\n    local_dir = Path(local_dir)\n    local_dir.mkdir(parents=True, exist_ok=True)\n    wanted = {r["id"]: r for r in selected_history}\n    protected_ids, protected_hashes = set(forbidden_ids), set(forbidden_hashes)\n    records, exclusions = {}, []\n    frozen, seen, package_receipts = [], set(), []\n    for item in package_items:\n        pkg = local_dir/item["arquivo"]\n        fetch_package(item, pkg)\n        if pkg.stat().st_size != item["zip_bytes"] or core.sha256_file(pkg) != item["zip_sha256"]:\n            raise RuntimeError("Pacote de replay incompleto ou alterado.")\n        with zipfile.ZipFile(pkg) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError("Lote de replay com membros repetidos.")\n            for name in z.namelist():\n                core.safe_member(name)\n            manifest = json.loads(z.read("manifesto_lote.json"))\n            if manifest["config_sha256"] != expected_cfg or manifest["origem_split"] != "train":\n                raise RuntimeError("Lote de replay não pertence ao TRAIN autenticado.")\n            for r in manifest["aceitas"]:\n                if r["id"] not in wanted:\n                    continue\n                if r["id"] in seen:\n                    raise RuntimeError("ID de replay repetido entre lotes.")\n                seen.add(r["id"])\n                prior = wanted[r["id"]]\n                if (prior["split"] != "train" or prior["lote"] != item["arquivo"] or\n                        prior["imagem_sha256"] != r["sha256"] or\n                        int(prior["known_mask"]) != r["known_mask"]):\n                    raise RuntimeError("Replay diverge da divisão 80/20 histórica.")\n                if r["id"] in protected_ids or r["sha256"] in protected_hashes:\n                    raise RuntimeError("Replay coincide com dado protegido.")\n                raw_image, raw_label, raw_cov = (z.read(r[k]) for k in ["imagem", "label", "cobertura"])\n                for raw, key in [(raw_image, "sha256"), (raw_label, "label_sha256"), (raw_cov, "cobertura_sha256")]:\n                    check_digest(raw, r[key], r["id"]+"/"+key)\n                classes = validator(raw_label, source=item["arquivo"]+":"+r["label"])\n                cov = json.loads(raw_cov)\n                known_classes = (set(cov["classes_positivas_verificadas_exaustivas"]) |\n                                 set(cov["classes_negativas_verificadas"]))\n                if (not all(isinstance(cid, int) and 0 <= cid < 12 for cid in known_classes)\n                        or sum(1 << cid for cid in known_classes) != r["known_mask"]\n                        or cov["known_mask"] != r["known_mask"]\n                        or set(classes) & set(cov["classes_negativas_verificadas"])):\n                    raise RuntimeError("Máscara ou anotação parcial de replay incompatível.")\n                image = local_dir/"images/train"/f"oi_{r[\'id\']}.jpg"\n                label = local_dir/"labels/train"/f"oi_{r[\'id\']}.txt"\n                coverage_file = local_dir/"coverage/train"/f"oi_{r[\'id\']}.json"\n                for path, raw in [(image, raw_image), (label, raw_label), (coverage_file, raw_cov)]:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(raw)\n                with Image.open(image) as im:\n                    width, height = im.size\n                    im.verify()\n                if (width, height) != (r["width"], r["height"]):\n                    raise RuntimeError("Dimensões de replay incompatíveis.")\n                near = protected_near.find(d11.phash(image))\n                if near:\n                    exclusions.append({"id": r["id"], "motivo": "semelhanca_phash_com_validacao_ou_reserva", "semelhante_a": near})\n                    continue\n                gt = core.parse_yolo(raw_label, width, height)\n                records[r["id"]] = {\n                    **prior, "oi_id": r["id"], "id": r["id"], "source": "openimages",\n                    "image": image, "label": label, "coverage": coverage_file,\n                    "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "label_sha256": r["label_sha256"], "coverage_sha256": r["cobertura_sha256"],\n                    "width": width, "height": height, "classes_com_caixa": classes,\n                    "gt": gt, "name": image.name, "role": "train", "coorte": "replay_train",\n                }\n        package_receipts.append(item)\n        pkg.unlink()\n    if seen != set(wanted):\n        raise RuntimeError("Alguns IDs planejados não foram encontrados nos cinco lotes.")\n    if len(records) < final_size:\n        raise RuntimeError(f"Replay com somente {len(records)} elegíveis; exigidos {final_size}.")\n    chosen = diverse_subset(list(records.values()), final_size, seed)\n    output = {Path(r["image"]).resolve().as_posix(): r for r in chosen}\n    for record in output.values():\n        frozen += [(record["image"], record["sha256"]), (record["label"], record["label_sha256"]),\n                   (record["coverage"], record["coverage_sha256"])]\n    return output, exclusions, frozen, package_receipts\n\n\ndef make_training_inputs(home, replay, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    home_train = {Path(r["image"]).resolve().as_posix():\n                  {**r, "source": "home", "known_mask": core.ALL_KNOWN}\n                  for r in home if r["role"] == "home_train"}\n    val = [Path(r["image"]).resolve().as_posix() for r in home if r["role"] == "home_val"]\n    if len(home_train) != 1824 or len(val) != 456:\n        raise RuntimeError("Home diferente da configuração congelada.")\n    val_file = destination/"home_val.txt"\n    val_file.write_text("\\n".join(sorted(val))+"\\n", encoding="utf-8")\n    arms = {}\n    for name, records in [("A_home", home_train), ("B_home_replay", {**home_train, **replay})]:\n        path = destination/(name+".txt")\n        path.write_text("\\n".join(sorted(records))+"\\n", encoding="utf-8")\n        data_file = destination/(name+".yaml")\n        data_file.write_text(yaml.safe_dump({"path": str(destination.resolve()), "train": str(path.resolve()),\n                                           "val": str(val_file.resolve()), "names": dict(enumerate(core.NAMES))},\n                                          sort_keys=False, allow_unicode=True), encoding="utf-8")\n        arms[name] = {"records": records, "yaml": data_file}\n    return arms\n\n\ndef restore_result(package_path, cohort, model, cache_dir):\n    cache_dir = Path(cache_dir)/cohort/model\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(package_path) as z:\n        prefix = cohort+"/"+model+"/"\n        summary = json.loads(z.read(prefix+"metricas.json"))\n        def rows(name):\n            return list(csv.DictReader(io.StringIO(z.read(prefix+name).decode("utf-8-sig"))))\n        images, events, classes = rows("alertas_por_imagem.csv"), rows("avaliacao_objetos.csv"), rows("metricas_por_classe.csv")\n        for row in images:\n            for k in ["tp_alerta", "fn_alerta", "fp_alerta", "alta_ref", "alta_sem_alerta", "tp_detector",\n                      "fn_detector", "fp_detector", "gt_avaliaveis", "gt_ignorados", "predicoes_ignoradas_conf_025"]:\n                row[k] = int(row[k])\n            for k in ["cobertura_completa_alertas", "ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        for row in events:\n            row["classe_id"] = int(row["classe_id"])\n            for k in ["ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        cache = cache_dir/"predicoes_conf_0001.jsonl.gz"\n        cache.write_bytes(z.read(prefix+"predicoes_conf_0001.jsonl.gz"))\n    return {"summary": summary, "images": images, "events": events, "classes": classes, "cache_file": cache}\n\n\ndef verify_frozen(files):\n    for path, sha in files:\n        if core.sha256_file(path) != sha:\n            raise RuntimeError("Arquivo protegido alterado: "+str(path))\n', 'exp12_training.py': '"""Treino finito e avaliação isolada dos checkpoints."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom contextlib import contextmanager\nfrom pathlib import Path\nimport math\nimport random\nimport hashlib\n\nimport numpy as np\nimport torch\nfrom ultralytics import YOLO\nfrom ultralytics.utils.metrics import ap_per_class\nfrom ultralytics.utils.torch_utils import unwrap_model\n\nimport diag11_core as core\nfrom exp10_partial import TrainerOpenImages, LoaderFinito, export_standard_checkpoint\nfrom exp12_protocol import BudgetSampler\n\n\ndef state_fingerprint(model):\n    digest = hashlib.sha256()\n    for name, value in sorted(unwrap_model(model).state_dict().items()):\n        digest.update(name.encode())\n        digest.update(str(tuple(value.shape)).encode())\n        digest.update(value.detach().cpu().float().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef start_epoch(trainer):\n    trainer.train_loader.sampler.set_epoch(trainer.epoch)\n    trainer.start_epoch_updates = trainer.actual_updates\n\n\ndef check_epoch(trainer):\n    expected = (trainer.epoch+1)*trainer.control_spec["batches"]\n    if trainer.actual_updates != expected or trainer.batches_seen != expected:\n        raise RuntimeError("Orçamento de atualizações ou batches diferente do planejado.")\n    spec = trainer.control_spec\n    expected_sources = {"home": expected*(spec["batch"]-spec["oi_per_batch"])}\n    if spec["oi_per_batch"]:\n        expected_sources["openimages"] = expected*spec["oi_per_batch"]\n    if dict(trainer.source_views) != expected_sources:\n        raise RuntimeError("Exposição efetiva por fonte diferente da proporção planejada.")\n\n\nclass ControlledTrainer(TrainerOpenImages):\n    control_spec = None\n\n    def __init__(self, *args, **kwargs):\n        self.actual_updates, self.batches_seen = 0, 0\n        self.source_views = Counter()\n        super().__init__(*args, **kwargs)\n        if self.control_spec is None:\n            raise ValueError("Especificação do braço não configurada.")\n        self.add_callback("on_train_epoch_start", start_epoch)\n        self.add_callback("on_train_epoch_end", check_epoch)\n\n    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):\n        if mode != "train":\n            return super().get_dataloader(dataset_path, batch_size, rank, mode)\n        spec = self.control_spec\n        if (batch_size != spec["batch"] or self.args.nbs != spec["batch"]\n                or self.args.amp or self.world_size > 1 or self.args.compile\n                or self.args.rect or self.args.warmup_epochs != 0):\n            raise RuntimeError("Batch, acumulação, precisão ou dispositivo mudou o orçamento fixado.")\n        if any(float(getattr(self.args, k, 0)) != 0 for k in\n               ["mosaic", "mixup", "cutmix", "copy_paste"]):\n            raise RuntimeError("Augmentação composta não é compatível com a máscara por imagem.")\n        dataset = self.build_dataset(dataset_path, mode, batch_size)\n        home, oi = [], []\n        for index, path in enumerate(dataset.im_files):\n            key = Path(path).resolve().as_posix()\n            if key not in self.source_records:\n                raise RuntimeError("Imagem sem proveniência no loader controlado.")\n            source = self.source_records[key]["source"]\n            if source == "home":\n                home.append(index)\n            elif source == "openimages":\n                oi.append(index)\n            else:\n                raise RuntimeError("Fonte de treino não reconhecida.")\n        if len(home) != spec["home_unique"] or len(oi) != spec["oi_unique"]:\n            raise RuntimeError("Contagem de imagens únicas do braço foi alterada.")\n        sampler = BudgetSampler(home, oi, batch_size, spec["batches"],\n                                spec["oi_per_batch"], self.args.seed)\n        return LoaderFinito(dataset, batch_size=batch_size, sampler=sampler,\n                            num_workers=self.args.workers,\n                            pin_memory=self.device.type == "cuda",\n                            collate_fn=dataset.collate_fn,\n                            generator=torch.Generator().manual_seed(self.args.seed),\n                            drop_last=False)\n\n    def preprocess_batch(self, batch):\n        if len(batch["im_file"]) != self.control_spec["batch"]:\n            raise RuntimeError("Batch incompleto ou redução automática de batch detectada.")\n        if self.accumulate != 1:\n            raise RuntimeError("A acumulação deixou de ser uma atualização por batch.")\n        result = super().preprocess_batch(batch)\n        self.batches_seen += 1\n        for path in batch["im_file"]:\n            self.source_views[self.source_records[Path(path).resolve().as_posix()]["source"]] += 1\n        return result\n\n    def optimizer_step(self):\n        if not torch.isfinite(self.loss.detach()).all():\n            raise FloatingPointError("Loss não finita; o braço foi interrompido.")\n        grads = [torch.isfinite(p.grad).all() for p in self.model.parameters() if p.grad is not None]\n        if grads and not torch.stack(grads).all():\n            raise FloatingPointError("Gradiente não finito; o braço foi interrompido.")\n        if any(not math.isclose(group["lr"], self.control_spec["lr0"], rel_tol=1e-8, abs_tol=1e-14)\n               for group in self.optimizer.param_groups):\n            raise RuntimeError("Taxa de aprendizado diferente da taxa constante predefinida.")\n        super().optimizer_step()\n        self.actual_updates += 1\n\n    def _handle_nan_recovery(self, epoch):\n        # A recuperação silenciosa repetiria épocas e invalidaria o orçamento.\n        if self.loss is not None and not torch.isfinite(self.loss.detach()).all():\n            raise FloatingPointError("Treino não finito; recuperação automática desativada neste controle.")\n        if self.fitness is not None and not math.isfinite(float(self.fitness)):\n            raise FloatingPointError("Fitness não finito.")\n        return False\n\n    def save_model(self):\n        for model in [unwrap_model(self.model), self.ema.ema]:\n            if not all(torch.isfinite(v).all() for v in model.state_dict().values()\n                       if v.is_floating_point()):\n                raise FloatingPointError("Peso não finito antes de salvar o checkpoint.")\n        return super().save_model()\n\n\n@contextmanager\ndef isolated_evaluation():\n    """Validação não altera os fluxos de aleatoriedade do treino."""\n    py_state, np_state = random.getstate(), np.random.get_state()\n    cpu_state = torch.get_rng_state()\n    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    threads = torch.get_num_threads()\n    cudnn = (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic)\n    deterministic = (torch.are_deterministic_algorithms_enabled(),\n                     torch.is_deterministic_algorithms_warn_only_enabled())\n    try:\n        yield\n    finally:\n        random.setstate(py_state)\n        np.random.set_state(np_state)\n        torch.set_rng_state(cpu_state)\n        if cuda_state is not None:\n            torch.cuda.set_rng_state_all(cuda_state)\n        torch.set_num_threads(threads)\n        torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic = cudnn\n        torch.use_deterministic_algorithms(deterministic[0], warn_only=deterministic[1])\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef evaluate_checkpoint(weight, checkpoint_id, cohorts, output_dir, policy, matcher, device):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with isolated_evaluation():\n        model = YOLO(str(weight))\n        if {int(k): str(v) for k, v in model.names.items()} != dict(enumerate(core.NAMES)):\n            raise RuntimeError("Classes do checkpoint incompatíveis com as validações.")\n        result = {}\n        for cohort, records in cohorts.items():\n            if any(r["role"] not in {"home_val", "val_nova"} for r in records):\n                raise RuntimeError("Reserva ou treino entrou na avaliação de checkpoint.")\n            result[cohort] = core.evaluate_model(\n                model, records, "candidato", output_dir/cohort, policy, matcher, device, ap_per_class)\n        del model\n    return result\n\n\ndef metric_entry(checkpoint_id, arm, epoch, updates, weight, evaluated):\n    return {"id": checkpoint_id, "arm": arm, "epoch": epoch, "updates": updates,\n            "weight": str(weight), "weight_sha256": core.sha256_file(weight),\n            "home": evaluated["home_val_historica"]["summary"],\n            "external": evaluated["openimages_val_nova"]["summary"]}\n\n\ndef make_checkpoint_callback(arm, cohorts, output_dir, policy, matcher, device,\n                             entries, results, backup):\n    output_dir = Path(output_dir)\n    done = set()\n\n    def after_save(trainer):\n        epoch = trainer.epoch+1\n        if epoch in done:\n            return\n        check_epoch(trainer)\n        checkpoint_id = f"{arm}_e{epoch:02}"\n        weight = output_dir/"pesos_checkpoints"/(checkpoint_id+".pt")\n        weight.parent.mkdir(parents=True, exist_ok=True)\n        export = export_standard_checkpoint(trainer.last, weight)\n        core.save_json(output_dir/"exports"/(checkpoint_id+".json"), export)\n        # Preserva o peso antes da avaliação mais demorada.\n        backup("checkpoint salvo: "+checkpoint_id)\n        evaluated = evaluate_checkpoint(weight, checkpoint_id, cohorts,\n                                        output_dir/"avaliacoes"/checkpoint_id,\n                                        policy, matcher, device)\n        entry = metric_entry(checkpoint_id, arm, epoch, trainer.actual_updates, weight, evaluated)\n        entries.append(entry)\n        results[checkpoint_id] = evaluated\n        core.save_json(output_dir/"checkpoints_avaliados.json", entries)\n        core.save_json(output_dir/"orcamento"/(checkpoint_id+".json"),\n                       {"arm": arm, "epoch": epoch, "updates": trainer.actual_updates,\n                        "batches_seen": trainer.batches_seen, "source_views": dict(trainer.source_views),\n                        "batch": trainer.control_spec["batch"], "amp": False, "accumulate": trainer.accumulate})\n        done.add(epoch)\n        backup("checkpoint avaliado: "+checkpoint_id)\n    return after_save\n', 'exp12_reports.py': '"""Relatório do controle 12 e galeria de rótulos sem alteração automática."""\nfrom __future__ import annotations\nimport html\nfrom pathlib import Path\nimport zipfile\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\n\nimport diag11_core as core\nimport diag11_reports as r11\nfrom exp12_protocol import order_id\n\n\ndef semantic_gallery(home, replay, output, policy, limit=12, seed=42):\n    output = Path(output)\n    rows, cards = [], []\n    for source, records in [(\'Home TRAIN\',[r for r in home if r[\'role\']==\'home_train\']),\n                            (\'Open Images TRAIN\',list(replay.values()))]:\n        candidates = [r for r in records if any(o[\'classe_id\']==7 for o in r[\'gt\'])]\n        candidates.sort(key=lambda r:order_id(r[\'id\'],seed))\n        for i, record in enumerate(candidates[:limit]):\n            if record[\'role\'] not in {\'home_train\',\'train\'}:\n                raise RuntimeError(\'A galeria de rótulos deve usar somente treino.\')\n            relative = f\'galeria_rotulos/{source.split()[0]}_{i:02}.jpg\'\n            path = output/relative\n            path.parent.mkdir(parents=True,exist_ok=True)\n            objects = core.with_policy(record[\'gt\'],record[\'width\'],record[\'height\'],policy)\n            r11.draw_panel(record,objects,source+\' / wardrobe\').save(path,quality=90)\n            rows.append({\'id\':record[\'id\'],\'fonte\':source,\'imagem\':record[\'name\'],\n                         \'arquivo\':relative,\'anotacao_parece_correta\':\'nao_revisado\',\n                         \'mesmo_conceito_de_guarda_roupa\':\'nao_revisado\',\n                         \'fotografia_real\':\'nao_revisado\',\'observacao\':\'\'})\n            cards.append(\'<article><h3>\'+html.escape(source)+\'</h3><img src="\'+relative+\'" alt="Rótulos originais do treino"><p>\'+html.escape(record[\'name\'])+\'</p></article>\')\n    core.save_csv(output/\'revisao_rotulos_wardrobe.csv\',rows)\n    page = \'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><title>Revisão de rótulos</title><style>body{font:16px system-ui;max-width:1200px;margin:30px auto;background:#f5f6f8}article{display:inline-block;vertical-align:top;margin:8px;background:white;padding:12px}img{width:500px;max-width:100%}</style><h1>Guarda-roupa: revisão dos dados de treino</h1><p>As caixas são as originais. Armários de parede, ilustrações e diferenças de significado entre bases precisam de revisão humana. Nenhum rótulo foi alterado neste experimento. Esta amostra por hash não estima a frequência de erros.</p>\'+\'\'.join(cards)+\'</html>\'\n    (output/\'revisao_rotulos.html\').write_text(page,encoding=\'utf-8\')\n    return rows\n\n\ndef checkpoint_rows(selection):\n    rows=[]\n    for entry in selection[\'assessed\']:\n        row={k:entry[k] for k in [\'id\',\'arm\',\'epoch\',\'updates\',\'weight_sha256\']}\n        for key,prefix in [(\'home\',\'home\'),(\'external\',\'externa\')]:\n            s=entry[key];a=s[\'alertas_por_objeto\']\n            row.update({prefix+\'_\'+k:a[k] for k in [\'tp\',\'fn\',\'fp\',\'precision\',\'recall\',\'f1\']})\n            row[prefix+\'_alta_sem_alerta\']=s[\'alta_sem_alerta\']\n            row[prefix+\'_alta_ref\']=s[\'alta_ref\']\n        row.update(entry[\'checks\']);rows.append(row)\n    return rows\n\n\ndef learning_curves(output, entries, baseline_home):\n    fig,axes=plt.subplots(3,2,figsize=(12,10),sharex=True,layout=\'constrained\')\n    colors={\'A_home\':\'#2264aa\',\'B_home_replay\':\'#df6b25\'}\n    initial=next(e for e in entries if e[\'updates\']==0)\n    for col,(cohort,title) in enumerate([(\'home\',\'Home VAL histórica\'),(\'external\',\'Validação externa de seleção\')]):\n        for arm in colors:\n            sequence=[initial]+sorted([e for e in entries if e[\'arm\']==arm],key=lambda e:e[\'updates\'])\n            x=[e[\'updates\'] for e in sequence]\n            for row,key in enumerate([\'precision\',\'recall\',\'alta_sem_alerta\']):\n                y=[e[cohort][key] if key==\'alta_sem_alerta\' else 100*e[cohort][\'alertas_por_objeto\'][key] for e in sequence]\n                axes[row,col].plot(x,y,\'o-\',color=colors[arm],label=arm,linewidth=1.8)\n                if cohort==\'home\':\n                    b=baseline_home[key] if key==\'alta_sem_alerta\' else 100*baseline_home[\'alertas_por_objeto\'][key]\n                    if arm==\'A_home\':axes[row,col].axhline(b,color=\'#697586\',linestyle=\'--\',label=\'Baseline Home\')\n        axes[0,col].set_title(title)\n        for row,label in enumerate([\'Precisão de alertas (%)\',\'Recall de alertas (%)\',\'Objetos de prioridade alta sem alerta\']):\n            axes[row,col].set_ylabel(label);axes[row,col].grid(alpha=.2)\n            low,high=axes[row,col].get_ylim()\n            axes[row,col].set_ylim(max(0,low),min(100,high) if row<2 else max(1,high))\n        axes[2,col].yaxis.set_major_locator(MaxNLocator(integer=True))\n        axes[2,col].set_xlim(0,max(1,max(e[\'updates\'] for e in entries)))\n        axes[2,col].set_xticks(sorted({e[\'updates\'] for e in entries}))\n        axes[2,col].set_xlabel(\'Atualizações do otimizador por braço\')\n        axes[0,col].legend(fontsize=8)\n    fig.suptitle(\'Ajustes a partir do candidato 10 · confiança 0,25 · mesmas validações\',fontsize=13)\n    fig.savefig(Path(output)/\'curvas_validacao.png\',dpi=160)\n    plt.close(fig)\n\n\ndef build_report(output, selection, evaluations, references, cohorts, policy, rule, audit, seed=42):\n    output=Path(output)\n    rows=checkpoint_rows(selection)\n    core.save_csv(output/\'comparacao_checkpoints.csv\',rows)\n    learning_curves(output,selection[\'assessed\'],references[\'baseline\'][\'home_val_historica\'][\'summary\'])\n    eligible,exploratory=selection[\'eligible\'],selection[\'exploratory\']\n    chosen=eligible or exploratory\n    if chosen is None:raise RuntimeError(\'Candidato inicial deve continuar disponível para inspeção.\')\n    final=evaluations[chosen[\'id\']]\n    bootstrap=[];gallery=[];cards=[];diagnostic=[]\n    for cohort,records in cohorts.items():\n        for reference_name, reference in references.items():\n            for row in r11.paired_bootstrap(reference[cohort][\'images\'],final[cohort][\'images\'],1000,seed):\n                bootstrap.append({\'coorte\':cohort,\'referencia\':reference_name,\'checkpoint\':chosen[\'id\'],**row})\n        pair={\'antigo\':references[\'baseline\'][cohort],\'candidato\':final[cohort]}\n        diagnostic.extend(r11.diagnosis_rows({cohort:pair}))\n        caches={k:r11.load_prediction_cache(v[\'cache_file\']) for k,v in pair.items()}\n        mapping={r[\'id\']:r for r in records}\n        cases=r11.select_gallery_cases(pair[\'antigo\'][\'images\'],pair[\'candidato\'][\'images\'],limit=12,seed=seed)\n        for i,case in enumerate(cases):\n            record=mapping[case[\'id\']];relative=f\'galeria/{cohort}_{i:02}.jpg\'\n            r11.render_case(record,caches[\'antigo\'][case[\'id\']],caches[\'candidato\'][case[\'id\']],policy,output/relative)\n            gallery.append({**case,\'coorte\':cohort,\'checkpoint\':chosen[\'id\'],\'arquivo\':relative,\'creditos\':record.get(\'creditos\',{})})\n            cards.append(\'<article><h3>\'+html.escape(cohort+\' / \'+case[\'categoria\'])+\'</h3><img loading="lazy" src="\'+relative+\'" alt="Referência, baseline e checkpoint"><p>Antigo = baseline. Candidato = \'+html.escape(chosen[\'id\'])+\'.</p></article>\')\n    core.save_csv(output/\'bootstrap_pareado.csv\',bootstrap)\n    core.save_csv(output/\'diagnostico_erros.csv\',diagnostic)\n    core.save_json(output/\'indice_galeria.json\',gallery)\n    core.save_json(output/\'selecao_checkpoints.json\',selection)\n    status=(\'Um checkpoint cumpriu todos os requisitos nas validações: \'+eligible[\'id\']+\'. Requer avaliação futura independente antes de substituir o modelo em uso.\' if eligible else \'Nenhum checkpoint cumpriu todos os requisitos. O baseline continua sendo a referência em uso. O melhor resultado exploratório é apresentado para diagnóstico.\')\n    header=[\'Checkpoint\',\'Atualizações\',\'P Home\',\'R Home\',\'Alta omitida Home\',\'P externa\',\'R externa\',\'Alta omitida externa\',\'Elegível\']\n    table=\'<table><tr>\'+\'\'.join(\'<th>\'+x+\'</th>\' for x in header)+\'</tr>\'\n    for r in rows:\n        values=[r[\'id\'],r[\'updates\'],f"{100*r[\'home_precision\']:.2f}%",f"{100*r[\'home_recall\']:.2f}%",r[\'home_alta_sem_alerta\'],f"{100*r[\'externa_precision\']:.2f}%",f"{100*r[\'externa_recall\']:.2f}%",r[\'externa_alta_sem_alerta\'],\'Sim\' if r[\'elegivel\'] else \'Não\']\n        table+=\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in values)+\'</tr>\'\n    table+=\'</table>\'\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>TCC — Experimento 12</title><style>body{font:16px/1.6 system-ui;color:#172335;background:#f3f5f8;margin:0}main{max-width:1400px;margin:auto;padding:30px}article,.note{background:white;border:1px solid #dbe2ec;border-radius:10px;padding:18px;margin:20px 0}img{max-width:100%;height:auto}table{border-collapse:collapse;width:100%;background:white}td,th{padding:9px;text-align:left;border:1px solid #dbe2ec}.scroll{overflow:auto}h1{line-height:1.2}</style><main><h1>Experimento 12 — Recuperar Home e preservar o ganho externo</h1>\'\'\'\n    page+=\'<div class="note"><strong>\'+html.escape(status)+\'</strong><p>Dois ajustes a partir do mesmo candidato 10. Orçamento de atualizações igual, exposição por fonte diferente e registrada. Política V2 e confiança 0,25 preservadas. As 300 imagens reservadas, Home TEST e Open Images TEST não foram inferidas.</p></div>\'\n    page+=\'<h2>Seleção em duas validações</h2><div class="scroll">\'+table+\'</div><img src="curvas_validacao.png" alt="Métricas nas duas validações por atualizações">\'\n    page+=\'<p>Requisitos Home: precisão maior que \'+f"{100*rule[\'home\'][\'precision_maior_que\']:.3f}%"+\', recall de pelo menos \'+f"{100*rule[\'home\'][\'recall_minimo\']:.3f}%"+\' e até \'+str(rule[\'home\'][\'alta_sem_alerta_maximo\'])+\' objetos de prioridade alta sem alerta. Requisitos externos: preservar as três métricas do candidato inicial. As referências foram medidas no mesmo ambiente antes dos novos treinos.</p>\'\n    page+=\'<h2>Limites da conclusão</h2><p>A validação externa passou a participar da seleção e deixou de ser uma avaliação final independente deste ajuste. Home VAL é histórica; a origem do primeiro treinamento continua \'+html.escape(audit[\'baseline_original\'][\'estado_proveniencia_original\'])+\'. Bootstrap sobre as validações é descritivo e não corrige seleção de checkpoints, múltiplas comparações ou agrupamento de imagens da mesma cena. Uma única semente e o orçamento curto não permitem concluir qual abordagem é superior em geral. Open Images tem anotações parciais e não garante cenas internas. Alertas medem concordância com a política nas caixas, sem validação de risco físico.</p>\'\n    page+=\'<p><a href="revisao_rotulos.html">Revisar guarda-roupas nos dados de treino</a>. O experimento mantém os rótulos originais. Uma divergência semântica confirmada exige uma correção documentada e um novo experimento.</p><h2>Casos para inspeção</h2><p>Checkpoint mostrado: \'+html.escape(chosen[\'id\'])+\'. A galeria prioriza diferenças e erros e não estima sua frequência.</p>\'+\'\'.join(cards)+\'</main></html>\'\n    (output/\'relatorio_experimento_12.html\').write_text(page,encoding=\'utf-8\')\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Experimento 12\\n\\n\'+status+\'\\n\\nCheckpoint na galeria: \'+chosen[\'id\']+\'.\\n\\nAbra relatorio_experimento_12.html e comparacao_checkpoints.csv. Os intervalos em bootstrap_pareado.csv são descritivos após seleção. Nenhum modelo em uso foi substituído. A reserva de 300 imagens permanece sem inferência.\\n\',encoding=\'utf-8\')\n    return chosen,status\n\n\ndef package_reports(output):\n    output=Path(output);archive=output/\'pacote_experimento_12.zip\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for path in sorted(output.rglob(\'*\')):\n            relative=path.relative_to(output)\n            if (not path.is_file() or path==archive or path.name.endswith(\'.tmp\')\n                    or path.suffix in {\'.pt\',\'.pkl\'} or relative.parts[0]==\'treinos\'\n                    or path.name==\'predicoes_conf_0001.jsonl.gz\'):\n                continue\n            z.write(path,relative)\n    return archive\n'}

PASTA_CODIGO=PASTA_LOCAL/'codigo'
PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    (PASTA_CODIGO/nome).write_text(fonte,encoding='utf-8')
for nome,esperado in EXPECTED['codigo_11'].items():
    if hashlib.sha256(MODULOS_EMBUTIDOS[nome].encode()).hexdigest()!=esperado:
        raise RuntimeError('O código de avaliação mudou em relação ao diagnóstico 11.')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import diag11_data as dados11
import exp10_partial as parcial
import exp12_protocol as protocolo
import exp12_data as dados
import exp12_training as treino
import exp12_reports as relatorios
parcial.check_api()
parcial.self_test_bce()
print('API, gradientes da perda parcial e avaliador histórico conferidos.')


## 3. Conectar ao Drive e autenticar os arquivos

Usa a conexão dos notebooks anteriores. Credenciais ficam em /tmp, fora
dos resultados. Pesos, divisão 80/20, pacote 11 e ZIP da validação nova
precisam coincidir com os hashes já avaliados. Não será criada outra validação.


In [ ]:
from pathlib import Path
import base64
import configparser
import json
import shutil
import subprocess
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

SECRET_DRIVE = "GDRIVE_RCLONE_CONFIG_B64"
REMOTE_DRIVE = "gdrive_tcc"
ORIGEM_DRIVE = f"{REMOTE_DRIVE}:TCC_Visao_Computacional"
DRIVE_TCC = Path("/tmp/tcc_100k_entradas")
ENVIAR_RESULTADOS_AO_DRIVE = True
EXECUTAR_CALIBRACAO = False

# Credenciais ficam fora da pasta de resultados salva pelo Kaggle.
PASTA_RCLONE = Path("/tmp/tcc_rclone")
PASTA_RCLONE.mkdir(mode=0o700, parents=True, exist_ok=True)
PASTA_RCLONE.chmod(0o700)
CONFIG_RCLONE = PASTA_RCLONE / "gdrive_tcc.conf"
RCLONE = PASTA_RCLONE / "rclone"

try:
    segredo = UserSecretsClient().get_secret(SECRET_DRIVE)
except Exception:
    raise RuntimeError(
        f"Crie o Secret {SECRET_DRIVE} em Add-ons → Secrets e habilite "
        "o acesso deste notebook. Siga a preparação descrita acima."
    ) from None
try:
    configuracao_bytes = base64.b64decode(segredo.strip(), validate=True)
    configuracao = configparser.ConfigParser(interpolation=None)
    configuracao.read_string(configuracao_bytes.decode("utf-8-sig"))
except Exception:
    raise RuntimeError(
        "O Secret não contém a configuração rclone codificada corretamente. "
        "Copie novamente pelo comando PowerShell e atualize o Secret."
    ) from None
if (not configuracao.has_section(REMOTE_DRIVE)
        or configuracao.get(REMOTE_DRIVE, "type", fallback="") != "drive"
        or not configuracao.get(REMOTE_DRIVE, "token", fallback="")
        or not configuracao.get(REMOTE_DRIVE, "client_id", fallback="")):
    raise RuntimeError(
        "A configuração precisa da conexão gdrive_tcc, tipo drive, "
        "cliente OAuth próprio e autorização concluída no navegador."
    )
if (ENVIAR_RESULTADOS_AO_DRIVE
        and configuracao.get(REMOTE_DRIVE, "scope", fallback="") != "drive"):
    raise RuntimeError(
        "Para salvar os resultados, configure scope 1 (drive, leitura e "
        "gravação) no rclone e atualize o Secret."
    )
# Esta conexão usa Meu Drive, não um Drive compartilhado.
configuracao.remove_option(REMOTE_DRIVE, "team_drive")
with CONFIG_RCLONE.open("w", encoding="utf-8") as arquivo_config:
    configuracao.write(arquivo_config)
CONFIG_RCLONE.chmod(0o600)
del segredo, configuracao_bytes, configuracao

if not RCLONE.is_file():
    import os

    executavel = shutil.which("rclone")
    if executavel is None:
        prefixo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(prefixo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            prefixo + ["apt-get", "install", "-y", "--no-install-recommends", "rclone"],
            check=True,
        )
        executavel = shutil.which("rclone")

    if executavel is None:
        raise RuntimeError("A instalação terminou, mas o executável rclone não foi localizado.")

    shutil.copy2(executavel, RCLONE)
    RCLONE.chmod(0o700)
    print("rclone instalado pelo gerenciador de pacotes do Linux.")

DRIVE_TCC.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(DRIVE_TCC).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB livres para baixar as entradas.")

def baixar_arquivo_drive(relativo, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    print("Baixando:", relativo, flush=True)
    concluido = subprocess.run(
        [str(RCLONE), "--config", str(CONFIG_RCLONE), "copyto",
         f"{ORIGEM_DRIVE}/{relativo}", str(destino),
         "--retries", "3", "--low-level-retries", "5"], check=False,
    )
    if concluido.returncode != 0 or not destino.is_file():
        raise RuntimeError(f"Falha ao baixar {relativo}. Confira a saída do rclone, a autorização e a Internet.")
    return destino


def enviar_resultados_drive(etapa="manual", obrigatorio=False):
    if not ENVIAR_RESULTADOS_AO_DRIVE:
        return False
    pasta = Path(PASTA_RESULTADOS)
    destino = f"{ORIGEM_DRIVE}/resultados/{NOME_EXPERIMENTO}"
    print(f"Enviando resultados ao Drive ({etapa})...", flush=True)
    try:
        enviado = subprocess.run(
            [str(RCLONE), "--config", str(CONFIG_RCLONE), "copy",
             str(pasta), destino, "--exclude", "**/*.tmp",
             "--retries", "2", "--low-level-retries", "3",
             "--contimeout", "15s", "--timeout", "60s"],
            check=False,
        )
        if enviado.returncode != 0:
            raise RuntimeError(f"rclone retornou código {enviado.returncode}")
    except (OSError, RuntimeError) as exc:
        mensagem = (
            f"Envio ao Drive falhou em {etapa}: {exc}. "
            "Os arquivos locais continuam em /kaggle/working/resultados_tcc. "
            "Confira a autorização e execute a última etapa para repetir o envio."
        )
        if obrigatorio:
            raise RuntimeError(mensagem) from None
        print(mensagem, flush=True)
        return False
    print("Resultados copiados ao Drive.", flush=True)
    return True

def enviar_resultados(etapa):
    return enviar_resultados_drive(etapa,obrigatorio=True)

ENTRADAS=PASTA_LOCAL/'entradas'
OI_RELATIVO='datasets/openimages_v7_tcc_100k_v1'
ZIP_HOME=baixar_arquivo_drive('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip')
PREP=baixar_arquivo_drive(OI_RELATIVO+'/configuracao_preparacao.json',ENTRADAS/'configuracao_preparacao.json')
PACOTES=baixar_arquivo_drive(OI_RELATIVO+'/manifesto_pacotes.json',ENTRADAS/'manifesto_pacotes.json')
MODELOS={
    'antigo':baixar_arquivo_drive('modelos/yolov8n_baseline/best.pt',ENTRADAS/'baseline.pt'),
    'candidato':baixar_arquivo_drive(PASTA_EXECUCAO_10+'/best.pt',ENTRADAS/'candidato_10.pt'),
}
DIVISAO_CSV=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.csv',ENTRADAS/'divisao_10.csv')
DIVISAO_JSON=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.json',ENTRADAS/'divisao_10.json')
PACOTE_11=baixar_arquivo_drive(PASTA_EXECUCAO_11+'/pacote_diagnostico_11.zip',ENTRADAS/'pacote_11.zip')
ZIP_NOVO=baixar_arquivo_drive(LOCALIZACAO_11['pasta_drive']+'/'+LOCALIZACAO_11['arquivo'],ENTRADAS/'validacao_11.zip')
for name,key in [('antigo','baseline'),('candidato','candidato')]:
    if core.sha256_file(MODELOS[name])!=EXPECTED[key]:
        raise RuntimeError('Peso diferente do avaliado: '+name)
cfg=json.loads(PREP.read_text())
if hashlib.sha256(core.canonical(cfg).encode()).hexdigest()!=EXPECTED['preparacao']:
    raise RuntimeError('O mapeamento de classes da preparação mudou.')
if core.sha256_file(PACOTES)!=EXPECTED['manifesto_pacotes']:
    raise RuntimeError('Manifesto dos pacotes diferente do histórico.')
HISTORICO=dados11.load_history(DIVISAO_CSV,DIVISAO_JSON,EXPECTED['divisao_10'])
RESUMO_11,CONFIG_11,LOCAL_11,MANIFESTO_NOVO=dados.read_package11(
    PACOTE_11,EXPECTED['pacote_11'],{'antigo':EXPECTED['baseline'],'candidato':EXPECTED['candidato']},EXPECTED['dataset_novo'])
if LOCAL_11!=LOCALIZACAO_11 or CONFIG_11['politica_sha256']!=EXPECTED['politica']:
    raise RuntimeError('Localização ou política do diagnóstico 11 diferente da esperada.')
FROZEN_FILES=[(p,core.sha256_file(p)) for p in [ZIP_HOME,PREP,PACOTES,DIVISAO_CSV,DIVISAO_JSON,PACOTE_11,ZIP_NOVO,*MODELOS.values()]]
print('Entradas e 100 mil IDs históricos conferidos.')


## 4. Recuperar Home e apenas a validação externa de seleção

Home TRAIN/VAL atual terá imagens e labels originais recuperados. Home TEST
não é extraído. A reserva externa permanece no ZIP; apenas seu manifesto
é consultado para bloquear coincidências de IDs, bytes e suspeitas por pHash.
A auditoria do treinamento original do baseline permanece como no diagnóstico 11.


In [ ]:
HOME,HASHES_HOME,PHASH_HOME,AUDITORIA_HOME=dados11.extract_home(
    ZIP_HOME,PASTA_LOCAL/'home',EXPECTED['home_zip'],EXPECTED['test_home'],EXPECTED['politica'])
dados.materialise_home_labels(ZIP_HOME,HOME)
sys.path.insert(0,str(PASTA_LOCAL/'home/codigo'))
import src.politica_alertas as politica
import src.validacao_alertas as avaliacao_base
if (core.sha256_file(politica.__file__)!=EXPECTED['politica'] or
    float(avaliacao_base.IOU_CORRESPONDENCIA)!=.5 or float(avaliacao_base.IOU_NMS)!=.7 or int(avaliacao_base.IMGSZ)!=640):
    raise RuntimeError('A política ou o associador mudou.')
VAL_NOVA,IDS_RESERVA,congelados=dados.load_frozen_validation(
    ZIP_NOVO,PASTA_LOCAL/'validacao',EXPECTED['zip_novo'],EXPECTED['dataset_novo'],MANIFESTO_NOVO)
FROZEN_FILES+=congelados
for record in HOME:
    FROZEN_FILES += [(record['image'],record['sha256']),(record['label'],record['label_sha256'])]
COORTES={'home_val_historica':[r for r in HOME if r['role']=='home_val'],
         'openimages_val_nova':VAL_NOVA}
novos=MANIFESTO_NOVO['arquivos']
if (set(r['id'] for r in novos)&HISTORICO['ids'] or
    set(r['sha256'] for r in novos)&(HISTORICO['hashes']|HASHES_HOME)):
    raise RuntimeError('A validação nova coincide com o histórico conhecido.')
if len(IDS_RESERVA)!=300 or any('reserva_futura' in p.parts for p in (PASTA_LOCAL/'validacao').rglob('*')):
    raise RuntimeError('Reserva extraída ou número inesperado de IDs.')
AUDITORIA={'separacao_atual':AUDITORIA_HOME,'baseline_original':RESUMO_11['auditoria']['baseline_original'],
           'reserva_futura_extraida':False,'reserva_futura_inferida':False,
           'test_home_inferido':False,'test_oi20k_inferido':False}
core.save_json(PASTA_RESULTADOS/'auditoria_dados.json',AUDITORIA)
print('Home: 1.824 TRAIN / 456 VAL. Externa: 1.000 VAL; reserva: 300 sem extração.')


## 5. Montar replay autenticado somente de TRAIN

Seleção por classe primária e hash, antes dos novos treinos e sem previsões.
Os cinco lotes têm 5.000 IDs já usados como treino no piloto, separados do TEST
histórico. Serão escolhidos 2.200 candidatos para permitir exclusões por pHash,
dos quais 2.000 formarão o replay. A diversidade usa pelo menos 40 imagens
por classe primária quando disponíveis; isso não representa frequência natural.

Bytes iguais a qualquer imagem Home ou ao conjunto externo de 1.300 imagens
são bloqueados. Suspeitas visuais de proximidade com validações/reserva são
excluídas conservadoramente; pHash não prova identidade de cena.


In [ ]:
proibidos_hash=HASHES_HOME|{r['sha256'] for r in novos}
selecionados=protocolo.choose_replay(HISTORICO['rows'],2200,SEED,40,proibidos_hash)
indice_protegido=dados11.NearIndex()
for record in HOME:
    if record['role']=='home_val':indice_protegido.add(record['phash'],record['id'])
for record in novos:indice_protegido.add(int(record['phash'],16),record['id'])
pacotes=json.loads(PACOTES.read_text())
if pacotes['config_sha256']!=EXPECTED['preparacao']:raise RuntimeError('Configuração dos lotes mudou.')
nomes_lotes={r['lote'] for r in selecionados}
lotes=sorted([r for r in pacotes['lotes'] if r['arquivo'] in nomes_lotes],key=lambda r:r['arquivo'])
if len(lotes)!=5:raise RuntimeError('Esperados os cinco lotes do piloto.')
def buscar_lote(item,destino):
    return baixar_arquivo_drive(OI_RELATIVO+'/lotes/'+item['arquivo'],destino)
ids_test={r['id'] for r in HISTORICO['rows'] if r['split']=='test'}
REPLAY,EXCLUSOES,congelados,RECIBOS=dados.load_replay(
    lotes,selecionados,PASTA_LOCAL/'replay',buscar_lote,EXPECTED['preparacao'],
    ids_test|{r['id'] for r in novos},proibidos_hash,indice_protegido,parcial.validate_yolo,
    final_size=2000,seed=SEED)
FROZEN_FILES+=congelados
BRACOS=dados.make_training_inputs(HOME,REPLAY,PASTA_LOCAL/'listas_treino')
core.save_json(PASTA_RESULTADOS/'manifesto_replay.json',{
    'criterio':'Diversidade por classe primária, ordem por hash, somente TRAIN dos cinco lotes do piloto',
    'imagens':[{k:r[k] for k in ['id','lote','classe_primaria','sha256','label_sha256','coverage_sha256','known_mask']} for r in REPLAY.values()],
    'exclusoes_phash':EXCLUSOES,'pacotes_conferidos':RECIBOS})
core.save_csv(PASTA_RESULTADOS/'replay_ids.csv',[{k:r[k] for k in ['id','lote','classe_primaria','known_mask','sha256']} for r in REPLAY.values()])
FROZEN_FILES += [(p,core.sha256_file(p)) for p in (PASTA_LOCAL/'listas_treino').glob('*') if p.is_file()]
REVISAO_ROTULOS=relatorios.semantic_gallery(HOME,REPLAY,PASTA_RESULTADOS,politica,seed=SEED)
print('Replay pronto:',len(REPLAY),'imagens TRAIN; exclusões por pHash:',len(EXCLUSOES))
print('Galeria dos rótulos originais:',PASTA_RESULTADOS/'revisao_rotulos.html')
enviar_resultados('dados e revisão dos rótulos')


## 6. Medir as referências e congelar os requisitos antes do treino

Baseline e candidato 10 são reavaliados no ambiente atual, usando o código
original e as mesmas imagens. As contagens serão comparadas com o pacote 11.
Diferenças entre CPU/GPU ficam registradas; denominadores diferentes interrompem
a execução. Os requisitos usam as referências atuais medidas **antes** dos ajustes.

Para ser elegível, um checkpoint precisa cumprir **todos** os requisitos:

- Home: precisão de alertas superior ao baseline, recall pelo menos igual e
  quantidade de objetos de prioridade alta sem alerta no máximo igual.
- Externa: precisão e recall de alertas pelo menos iguais ao candidato 10,
  com objetos de prioridade alta sem alerta no máximo iguais.

Confiança permanece 0,25. Não há calibração de limiar. Se nenhum checkpoint
cumprir tudo, o resultado é inconclusivo para substituição e o baseline segue
preservado. Não haverá relaxamento automático dos limites.


In [ ]:
REFERENCIAS={}
REPRODUCAO=[]
for nome,chave in [('baseline','antigo'),('candidato_10','candidato')]:
    print('Reavaliando referência:',nome,flush=True)
    REFERENCIAS[nome]=treino.evaluate_checkpoint(
        MODELOS[chave],nome,COORTES,PASTA_RESULTADOS/'referencias'/nome,
        politica,avaliacao_base.corresponder_objetos,DEVICE)
    for coorte,result in REFERENCIAS[nome].items():
        original=next(r for r in RESUMO_11['comparacao'] if r['coorte']==coorte and r['modelo']==chave)
        atual=result['summary'];a=atual['alertas_por_objeto']
        if (atual['imagens']!=original['imagens'] or atual['alta_ref']!=original['alta_ref'] or
            a['tp']+a['fn']!=original['tp_alerta']+original['fn_alerta']):
            raise RuntimeError('Os denominadores mudaram em relação ao diagnóstico 11.')
        for metrica,valor in [('tp_alerta',a['tp']),('fn_alerta',a['fn']),('fp_alerta',a['fp']),('alta_sem_alerta',atual['alta_sem_alerta'])]:
            REPRODUCAO.append({'modelo':nome,'coorte':coorte,'metrica':metrica,
                               'notebook11':original[metrica],'atual':valor,'delta':valor-original[metrica]})
    enviar_resultados('referência medida: '+nome)
core.save_csv(PASTA_RESULTADOS/'reproducao_referencias_11.csv',REPRODUCAO)
REGRA=protocolo.make_rule(REFERENCIAS['baseline']['home_val_historica']['summary'],
    REFERENCIAS['candidato_10']['openimages_val_nova']['summary'],EXPECTED['dataset_novo'],EXPECTED['politica'])
core.save_json(PASTA_RESULTADOS/'regra_selecao_congelada.json',REGRA)
HASH_REGRA=core.sha256_file(PASTA_RESULTADOS/'regra_selecao_congelada.json')
FROZEN_FILES.append((PASTA_RESULTADOS/'regra_selecao_congelada.json',HASH_REGRA))
ARGUMENTOS_TREINO=dict(epochs=EPOCAS,imgsz=640,batch=BATCH,nbs=BATCH,
    workers=WORKERS,device=DEVICE,amp=False,optimizer='AdamW',lr0=LR0,lrf=1.,cos_lr=False,
    momentum=.937,weight_decay=.0005,warmup_epochs=0.,mosaic=0.,mixup=0.,cutmix=0.,copy_paste=0.,
    close_mosaic=0,scale=.2,translate=.1,fliplr=.5,flipud=0.,degrees=0.,shear=0.,perspective=0.,
    hsv_h=.015,hsv_s=.7,hsv_v=.4,multi_scale=0.,pretrained=True,rect=False,compile=False,
    seed=SEED,deterministic=True,patience=0,save=True,save_period=1,plots=False,cache=False,
    val=True,conf=.001,iou=.7,max_det=300,resume=False,exist_ok=False,verbose=False)
CONFIG_12={'versao':VERSAO,'experimento':NOME_EXPERIMENTO,'expected':EXPECTED,
    'semente':SEED,'argumentos_treino':ARGUMENTOS_TREINO,'regra_sha256':HASH_REGRA,
    'epocas_curtas':EPOCAS,'batches_por_epoca':BATCHES_POR_EPOCA,'atualizacoes_por_braco':342,
    'bracos':protocolo.ARMS,'replay_unico':2000,'home_train_unico':1824,
    'exposicoes_planejadas':{'A_home':{'home':5472,'openimages':0},'B_home_replay':{'home':4104,'openimages':1368}},
    'mesmo_numero_atualizacoes_nao_mesma_exposicao_home':True,
    'validacao_externa_agora_usada_para_selecao':True,'reserva_futura_inferida':False,
    'rotulos_originais_preservados':True,'revisao_semantica':'pendente; galeria somente de treino',
    'ambiente':{'python':platform.python_version(),'torch':torch.__version__,'ultralytics':ultralytics.__version__,'device':str(DEVICE)},
    'codigo_embutido_sha256':{n:hashlib.sha256(s.encode()).hexdigest() for n,s in MODULOS_EMBUTIDOS.items()}}
core.save_json(PASTA_RESULTADOS/'configuracao_experimento_12.json',CONFIG_12)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    path=PASTA_RESULTADOS/'codigo'/nome;path.parent.mkdir(exist_ok=True);path.write_text(fonte,encoding='utf-8')
ENTRIES=[treino.metric_entry('candidato_10_inicial','inicial',0,0,MODELOS['candidato'],REFERENCIAS['candidato_10'])]
AVALIACOES={'candidato_10_inicial':REFERENCIAS['candidato_10']}
ORCAMENTOS=[]
TREINO_INICIADO=False
core.save_json(PASTA_RESULTADOS/'checkpoints_avaliados.json',ENTRIES)
display(pd.DataFrame(REPRODUCAO))
print('Regra congelada antes do treino:',HASH_REGRA)
print(json.dumps(REGRA,ensure_ascii=False,indent=2))
enviar_resultados('referências e protocolo congelados')


## 7. Executar os dois ajustes a partir do mesmo candidato

A e B começam separadamente no candidato 10. O estado inicial é conferido
depois de configurar cada treinador. Cada época terá exatamente 114 atualizações.
Cada checkpoint EMA é exportado como YOLO padrão, copiado ao Drive e avaliado
nas duas validações. As previsões preservam os estados aleatórios do treino.

O best.pt interno considera a validação padrão do YOLO e **não governa a seleção
deste experimento**. Aqui todos os três checkpoints por braço são comparados
pelos requisitos congelados de alertas. Loss, gradientes ou pesos não finitos,
redução automática de batch e divergência de orçamento interrompem o braço.


In [ ]:
if TREINO_INICIADO or ORCAMENTOS or len(ENTRIES)!=1:
    raise RuntimeError('Esta célula já iniciou. Para repetir, inicie uma nova execução desde o começo.')
TREINO_INICIADO=True
try:
    for arm,ratio in protocolo.ARMS.items():
        spec={'batch':BATCH,'batches':BATCHES_POR_EPOCA,'oi_per_batch':ratio['oi_per_batch'],
              'home_unique':1824,'oi_unique':2000 if ratio['oi_per_batch'] else 0,'lr0':LR0}
        TrainerBraco=type('Trainer_'+arm,(treino.ControlledTrainer,),{
            'source_records':BRACOS[arm]['records'],'control_spec':spec})
        model=YOLO(str(MODELOS['candidato']))
        fingerprint_inicial=treino.state_fingerprint(model.model)
        def conferir_inicio(trainer,arm=arm,esperado=fingerprint_inicial):
            observado=treino.state_fingerprint(trainer.model)
            if observado!=esperado:
                raise RuntimeError('Estado inicial diferente do candidato 10 no braço '+arm)
            core.save_json(PASTA_RESULTADOS/'inicializacoes'/(arm+'.json'),{
                'modelo_arquivo_sha256':EXPECTED['candidato'],'estado_inicial_sha256':observado,
                'mesmo_ponto_de_partida':True})
        model.add_callback('on_train_start',conferir_inicio)
        model.add_callback('on_model_save',treino.make_checkpoint_callback(
            arm,COORTES,PASTA_RESULTADOS,politica,avaliacao_base.corresponder_objetos,
            DEVICE,ENTRIES,AVALIACOES,enviar_resultados))
        print('Iniciando:',arm,'| batch:',BATCH,'| atualizações:',EPOCAS*BATCHES_POR_EPOCA,flush=True)
        model.train(trainer=TrainerBraco,data=str(BRACOS[arm]['yaml']),
            project=str(PASTA_RESULTADOS/'treinos'),name=arm,**ARGUMENTOS_TREINO)
        trainer=model.trainer
        treino.check_epoch(trainer)
        if trainer.actual_updates!=342 or len([r for r in ENTRIES if r['arm']==arm])!=EPOCAS:
            raise RuntimeError('Braço terminou fora do orçamento ou faltou avaliar checkpoint.')
        ORCAMENTOS.append({'arm':arm,'atualizacoes':trainer.actual_updates,
            'batches':trainer.batches_seen,'exposicoes':dict(trainer.source_views),'epocas':EPOCAS})
        core.save_json(PASTA_RESULTADOS/'orcamento_realizado.json',ORCAMENTOS)
        enviar_resultados('braço concluído: '+arm)
        del model,trainer
        if torch.cuda.is_available():torch.cuda.empty_cache()
except Exception as exc:
    core.save_json(PASTA_RESULTADOS/'falha_experimento_12.json',{
        'concluido':False,'erro':str(exc),'traceback':traceback.format_exc(),
        'checkpoints_avaliados':[r['id'] for r in ENTRIES],'orcamentos_completos':ORCAMENTOS})
    try:enviar_resultados('falha; preservando resultados disponíveis')
    except Exception:print('Confira também os resultados locais em /kaggle/working/resultados_tcc.')
    raise


## 8. Aplicar a regra, conferir integridade e gerar o pacote

Um checkpoint elegível é somente uma **proposta baseada nas validações**.
Não há promoção automática. Se nenhum cumprir os requisitos, a comparação
exploratória usa o checkpoint que melhor recupera recall Home entre os que
preservam as três métricas externas; o candidato inicial também participa.

As 1.000 imagens externas agora são usadas para selecionar checkpoints e,
portanto, não fornecem uma avaliação final independente desses ajustes.
O bootstrap das validações é descritivo após seleção. As 300 reservadas
continuam guardadas para uma decisão posterior, com checkpoint já congelado.


In [ ]:
if len(ORCAMENTOS)!=2 or len(ENTRIES)!=7:
    raise RuntimeError('Os dois braços e seus seis checkpoints precisam estar completos.')
dados.verify_frozen(FROZEN_FILES)
if hashlib.sha256(core.canonical(REGRA).encode()).hexdigest()!=hashlib.sha256(core.canonical(json.loads((PASTA_RESULTADOS/'regra_selecao_congelada.json').read_text())).encode()).hexdigest():
    raise RuntimeError('A regra em memória diverge da regra congelada.')
SELECAO=protocolo.select_checkpoints(ENTRIES,REGRA)
ESCOLHIDO,STATUS=relatorios.build_report(PASTA_RESULTADOS,SELECAO,AVALIACOES,
    REFERENCIAS,COORTES,politica,REGRA,AUDITORIA,SEED)
for entry,nome in [(SELECAO['eligible'],'modelo_proposto_validacao.pt'),
                   (SELECAO['exploratory'],'modelo_exploratorio.pt')]:
    if entry:
        destino=PASTA_RESULTADOS/nome
        shutil.copy2(entry['weight'],destino)
        if core.sha256_file(destino)!=entry['weight_sha256']:raise RuntimeError('Peso exportado diverge do avaliado.')
core.save_json(PASTA_RESULTADOS/'resumo_experimento_12.json',{
    'versao':VERSAO,'experimento':NOME_EXPERIMENTO,'concluido':True,
    'integridade_conferida':True,'novos_treinamentos':2,'orcamentos':ORCAMENTOS,
    'status':STATUS,'elegivel':SELECAO['eligible']['id'] if SELECAO['eligible'] else None,
    'exploratorio':SELECAO['exploratory']['id'] if SELECAO['exploratory'] else None,
    'checkpoint_galeria':ESCOLHIDO['id'],'dataset_sha256':EXPECTED['dataset_novo'],
    'regra_sha256':HASH_REGRA,'modelo_em_uso_continua_baseline':True,
    'modelos_substituidos':False,'reserva_futura_extraida':False,'reserva_futura_inferida':False,
    'test_home_inferido':False,'test_oi20k_inferido':False,'validacao_externa_usada_na_selecao':True,
    'rotulos_originais_preservados':True,'revisao_semantica_pendente':True})
PACOTE_FINAL=relatorios.package_reports(PASTA_RESULTADOS)
enviar_resultados('resultado final e pacote para análise')
print(STATUS)
display(pd.DataFrame(relatorios.checkpoint_rows(SELECAO)))
print('Pacote para enviar:',PACOTE_FINAL)
print('Os pesos ficam separados do pacote compacto, nos resultados desta execução.')
display(FileLink(str(PACOTE_FINAL)))
display(FileLink(str(PASTA_RESULTADOS/'relatorio_experimento_12.html')))


## 9. Arquivos para enviar de volta

Resultados em **pacote_experimento_12.zip**. Ele reúne comparação dos sete pontos
(candidato inicial + seis checkpoints), regra congelada, orçamentos reais,
auditoria, referências, curvas, galerias e relatório. Os pesos ficam no Drive
em resultados/ajuste12_.../pesos_checkpoints e nas saídas do Kaggle.

Em caso de interrupção, consultar o arquivo falha_experimento_12.json e a saída
da célula que falhou. Para repetir o experimento inteiro, comece uma nova
execução pela primeira célula; este notebook não retoma otimizadores parciais.
